<a href="https://colab.research.google.com/github/farrelrassya/LLM-Design-Patterns-Building-Robust-and-Efficient-AI-Systems/blob/main/15.Cross_Validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 15 -- Cross-Validation

*LLM Design Patterns*

**Technical edition, versi GPU untuk Colab -- kode buku plus eksperimen yang menguji klaimnya.**

Misalkan model meraih akurasi 0,78 pada test set. Seberapa yakin kita bahwa angka itu bertahan pada data lain? **Cross-validation** menjawabnya dengan membagi data menjadi beberapa bagian, lalu melatih dan menguji model berulang kali dengan bagian yang bergantian.

Contoh buku yang gagal dijalankan ada di kotak **Listing buku** beserta penjelasan error-nya, diikuti versi yang berjalan. Empat bagian **Under the hood** (UTH) adalah tambahan notebook ini:

| Under the hood | Yang diukur | Hasil (run acuan) |
|---|---|---|
| UTH-1: satu split atau k-fold? | 200 split tunggal lawan 40 kali 5-fold | varians 5-fold 11 kali lebih kecil |
| UTH-2: satu angka few-shot hanyalah satu sampel | 14 prompt GPT-2 pada SST-2 | akurasi 0,508-0,714; zero-shot 0,694 |
| UTH-3: lupa, atau salah membaca? | continual learning dan probe linear | probe tetap 0,813 walau model menebak satu kelas |
| UTH-4: kebocoran yang tak terlihat oleh split | near-duplicate; "penghafal" QQP | penghafal tanpa parameter 0,850, baseline 0,632 |

> **Catatan versi GPU.** Pilih *Runtime > Change runtime type > T4 GPU*, lalu *Runtime > Run all*. Unduhan pertama sekitar 4,5 GB. Edisi ini belum pernah dijalankan di GPU NVIDIA maupun di Colab, jadi semua sel kode masih kosong sampai dijalankan.
>
> Angka dan kutipan output di narasi berasal dari **run acuan** di CPU sebuah Apple M1 Pro (15 Oktober 2026), dengan versi paket yang meniru runtime Colab (Python 3.13, torch 2.11, transformers 5.18, datasets 4.8.5). Bagian yang tetap di CPU mestinya memberi angka yang sama, dan skor GPT-2 serta fitur BERT biasanya hanya bergeser di digit terakhir. Hasil fine-tuning BERT, di bagian adaptasi domain dan continual learning, bisa berbeda nyata, termasuk seed mana yang runtuh; di sana, cocokkan polanya, bukan angka persisnya.

## Persiapan

Sel pertama memastikan GPU benar-benar siap sebelum ada unduhan atau pelatihan. Ia tidak sekadar bertanya apakah **CUDA**, platform NVIDIA yang dipakai PyTorch untuk berhitung di GPU, tersedia, tetapi menyuruh GPU benar-benar mengalikan matriks dan menunggu hasilnya. Bila GPU tidak tersedia atau gagal berhitung, notebook berhenti di sini dengan petunjuk, bukan diam-diam pindah ke CPU. Tiga sel kode berikutnya menyiapkan paket, data, dan korpus yang dipakai sepanjang bab.

In [1]:
import os

# Harus sebelum import torch: torch ikut memuat tqdm, dan tqdm hanya membaca TQDM_DISABLE saat pertama dimuat.
os.environ.setdefault("TQDM_DISABLE", "1")                    # progress bar dimatikan agar output ringkas
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
import torch

# Edisi GPU: bab ini meminta CUDA. Ganti "cuda" menjadi "cpu" hanya bila sengaja menjalankan seluruh bab di CPU (jauh lebih lama).
DEVICE_REQUEST = os.environ.get("CH15_DEVICE", "cuda")
GPU_HELP = ("Di Colab: Runtime > Change runtime type > pilih T4 GPU (atau GPU lain) > Save, lalu Runtime > Run all. "
            'Untuk sengaja memakai CPU, ubah "cuda" di baris DEVICE_REQUEST menjadi "cpu".')
GPU_RESET_HELP = ("Di Colab: Runtime > Disconnect and delete runtime, sambungkan lagi dengan GPU, lalu Runtime > Run all. "
                  "Bila tetap gagal, coba lagi beberapa saat kemudian atau pilih jenis GPU lain.")


def prepare_device(request):
    """Kembalikan perangkat yang benar-benar siap. GPU yang diminta tetapi tidak berfungsi menghentikan notebook di sini."""
    request = request.strip().lower()
    if request == "cpu":
        return torch.device("cpu")
    if request == "mps":  # hanya untuk uji lokal di Mac, bukan untuk Colab
        if not torch.backends.mps.is_available():
            raise RuntimeError("MPS diminta, tetapi tidak tersedia di mesin ini.")
        return torch.device("mps")
    if request != "cuda":
        raise ValueError(f'DEVICE_REQUEST harus "cuda" atau "cpu", bukan {request!r}.')
    if not torch.cuda.is_available():
        raise RuntimeError("GPU diminta, tetapi CUDA tidak tersedia. " + GPU_HELP)
    try:  # alokasi dan perkalian matriks sungguhan, ditunggu sampai selesai
        probe = torch.randn(1024, 1024, device="cuda")
        checksum = (probe @ probe).sum()
        torch.cuda.synchronize()
        healthy = bool(torch.isfinite(checksum))
    except Exception as err:  # driver atau runtime CUDA yang rusak tidak selalu melempar RuntimeError
        raise RuntimeError("GPU terdeteksi, tetapi gagal menjalankan perhitungan uji. " + GPU_RESET_HELP) from err
    if not healthy:
        raise RuntimeError("GPU terdeteksi, tetapi perhitungan uji menghasilkan NaN atau inf. " + GPU_RESET_HELP)
    del probe, checksum
    torch.cuda.empty_cache()
    return torch.device("cuda")


DEVICE = prepare_device(DEVICE_REQUEST)
USE_CPU = DEVICE.type == "cpu"
if DEVICE.type == "cuda":
    major, minor = torch.cuda.get_device_capability(DEVICE)
    free_bytes, total_bytes = torch.cuda.mem_get_info(DEVICE)
    print(f"perangkat: {DEVICE} | {torch.cuda.get_device_name(DEVICE)} | compute capability {major}.{minor}"
          f" | VRAM {total_bytes / 1e9:.1f} GB, bebas {free_bytes / 1e9:.1f} GB | torch {torch.__version__}")
    print("presisi: FP32 untuk semua model" + ("; GPU ini juga mendukung BF16, tetapi tidak dipakai" if major >= 8
                                               else "; GPU ini tidak punya BF16 asli (T4 = 7.5), jadi BF16 tidak dipakai"))
else:
    print(f"perangkat: {DEVICE} (dipilih lewat DEVICE_REQUEST); bagian GPU akan berjalan jauh lebih lama")

perangkat: cuda | Tesla T4 | compute capability 7.5 | VRAM 15.6 GB, bebas 15.5 GB | torch 2.11.0+cu130
presisi: FP32 untuk semua model; GPU ini tidak punya BF16 asli (T4 = 7.5), jadi BF16 tidak dipakai


Bila GPU siap, sel di atas mencetak dua baris. Di Colab dengan T4, baris pertama kira-kira `perangkat: cuda | Tesla T4 | compute capability 7.5 | VRAM 15.8 GB, bebas ...`. **VRAM** adalah memori milik GPU itu sendiri, terpisah dari RAM Colab: setiap model yang dihitung di GPU harus muat di sana. Model terbesar bab ini, `gpt2-large`, memakai sekitar 3 GB.

Baris kedua menyangkut **presisi numerik**: berapa bit yang dipakai untuk menyimpan setiap angka (jangan tertukar dengan *presisi* sebagai metrik klasifikasi, yang muncul nanti di bagian CommitmentBank). Bawaan PyTorch adalah FP32 (32 bit). Format 16 bit seperti FP16 dan BF16 lebih hemat memori dan sering lebih cepat, tetapi pembulatannya lebih kasar. **Compute capability** adalah nomor generasi GPU NVIDIA; T4 bernomor 7.5 dan tidak punya BF16 asli di perangkat kerasnya. **Bab ini memakai FP32 di semua model, agar angkanya sebanding dengan angka di narasi, apa pun GPU yang kebetulan didapat.** VRAM T4 lebih dari cukup untuk itu.

Sebagian bab memang tidak butuh GPU: pembagian data, TF-IDF, dan deduplikasi memakai pandas, scikit-learn, dan datasketch, yang semuanya bekerja di CPU. Kepala linear di bagian CommitmentBank, masing-masing hanya 4.611 parameter, juga sengaja dilatih di CPU karena terlalu kecil untuk diuntungkan GPU. Sel komputasi utama bagian-bagian itu diawali komentar `# CPU` beserta alasannya.

In [2]:
import importlib.util, subprocess, sys

# datasketch (MinHash untuk deduplikasi) satu-satunya paket bab ini yang belum ada di runtime Colab.
if importlib.util.find_spec("datasketch") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "datasketch==2.0.0"])

def read_hf_token():
    """HF_TOKEN opsional, dari Colab Secrets atau environment. Nilainya tidak dicetak dan tidak ditulis ke file."""
    try:
        from google.colab import userdata
        return userdata.get("HF_TOKEN")
    except Exception:  # bukan Colab, secret belum dibuat, atau akses notebook ditolak
        return os.environ.get("HF_TOKEN")


token = read_hf_token()
if token:
    os.environ["HF_TOKEN"] = token
del token

import copy, gc, gzip, json, urllib.request
import numpy as np
import pandas as pd
import torch.nn.functional as F
import datasets, huggingface_hub, sklearn, transformers
from datasets import load_dataset

transformers.logging.set_verbosity_error()
transformers.logging.disable_progress_bar()
datasets.logging.set_verbosity_error()
datasets.disable_progress_bars()
huggingface_hub.utils.logging.set_verbosity_error()  # status HF_TOKEN sudah dicetak di bawah; peringatan server tak perlu diulang


def free_memory():
    """Lepaskan model dan tensor yang sudah di-del, termasuk cache memori GPU, sebelum eksperimen berikutnya."""
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    elif DEVICE.type == "mps":
        torch.mps.empty_cache()


def model_device(model, name):
    """Perangkat tempat model benar-benar berada. Menghentikan notebook bila ada parameter di luar perangkat
    yang diminta, atau bila presisinya bukan FP32."""
    params = list(model.parameters())
    stray = {str(p.device) for p in params if p.device.type != DEVICE.type}
    if stray:
        raise RuntimeError(f"{name} punya parameter di {sorted(stray)}, padahal yang diminta {DEVICE}.")
    dtypes = {str(p.dtype) for p in params}
    if dtypes != {"torch.float32"}:
        raise RuntimeError(f"{name} memakai {sorted(dtypes)}; edisi ini mengandalkan FP32.")
    return str(params[0].device)


def memory_report():
    """Memori GPU yang masih terpakai, dan puncaknya sejak laporan sebelumnya (puncaknya lalu di-reset)."""
    if DEVICE.type == "cuda":
        report = (f"VRAM terpakai {torch.cuda.memory_allocated() / 1e9:.2f} GB"
                  f" (puncak sejak laporan sebelumnya {torch.cuda.max_memory_allocated() / 1e9:.2f} GB)")
        torch.cuda.reset_peak_memory_stats()
        return report
    if DEVICE.type == "mps":
        return f"memori MPS terpakai {torch.mps.current_allocated_memory() / 1e9:.2f} GB"
    return "CPU: tidak ada memori GPU"


print("torch", torch.__version__, "| transformers", transformers.__version__, "| datasets", datasets.__version__,
      "| scikit-learn", sklearn.__version__, "| pandas", pd.__version__)
print("HF_TOKEN:", "terpasang" if os.environ.get("HF_TOKEN") else "tidak ada (unduhan anonim)")

torch 2.11.0+cu130 | transformers 5.20.2 | datasets 4.8.5 | scikit-learn 1.6.1 | pandas 2.2.3
HF_TOKEN: terpasang


Baris kedua output menunjukkan `HF_TOKEN: tidak ada`, dan itu tidak masalah: semua dataset dan model bab ini publik, jadi unduhan anonim sudah cukup. Bila token tersimpan di Colab Secrets, sel ini memakainya tanpa mencetak nilainya.

Sel ini juga menyiapkan tiga pembantu. `free_memory()` melepas memori GPU setiap kali sebuah model selesai dipakai. `model_device()` menghentikan notebook bila model tidak berada di perangkat yang diminta atau tidak dalam FP32. `memory_report()` mencetak VRAM yang masih terpakai dan puncaknya sejak laporan sebelumnya.

In [3]:
from huggingface_hub import hf_hub_download


def fetch(url, path):
    """Unduh sekali. File sementara di-rename di akhir, jadi unduhan yang terputus tidak meninggalkan file rusak."""
    if not os.path.exists(path):
        urllib.request.urlretrieve(url, path + ".part")
        os.replace(path + ".part", path)
    return path


os.makedirs("data", exist_ok=True)
GITHUB_FILES = {
    "tweeteval_sentiment_train_text.txt": "cardiffnlp/tweeteval/main/datasets/sentiment/train_text.txt",
    "tweeteval_sentiment_train_labels.txt": "cardiffnlp/tweeteval/main/datasets/sentiment/train_labels.txt",
    "tweeteval_sentiment_test_text.txt": "cardiffnlp/tweeteval/main/datasets/sentiment/test_text.txt",
    "tweeteval_sentiment_test_labels.txt": "cardiffnlp/tweeteval/main/datasets/sentiment/test_labels.txt",
    "gsm8k_train.jsonl": "openai/grade-school-math/master/grade_school_math/data/train.jsonl",
    "gsm8k_test.jsonl": "openai/grade-school-math/master/grade_school_math/data/test.jsonl",
    "HumanEval.jsonl.gz": "openai/human-eval/master/data/HumanEval.jsonl.gz",
    "mt_bench_question.jsonl": "lm-sys/FastChat/main/fastchat/llm_judge/data/mt_bench/question.jsonl",
}
for name, path in GITHUB_FILES.items():
    fetch(f"https://raw.githubusercontent.com/{path}", f"data/{name}")
    print(f"data/{name:<38} {os.path.getsize(f'data/{name}') / 1e3:8.1f} KB")

# TweetTopic: repo HF-nya berbasis skrip (ditolak load_dataset versi baru), jadi file JSON-nya diunduh langsung.
TT_SPLITS = ["train_2020", "validation_2020", "test_2020", "train_2021", "validation_2021", "test_2021"]
TT_FILES = {s: hf_hub_download("cardiffnlp/tweet_topic_single", f"dataset/split_temporal/{s}.single.json",
                               repo_type="dataset") for s in TT_SPLITS}
tweet_topic = pd.concat([pd.read_json(TT_FILES[s], lines=True, convert_dates=False) for s in TT_SPLITS],
                        ignore_index=True)
print(f"\nTweetTopic: {len(tweet_topic)} tweet, {tweet_topic['date'].min()} s.d. {tweet_topic['date'].max()}, "
      f"{tweet_topic['label_name'].nunique()} topik")

data/tweeteval_sentiment_train_text.txt       4970.0 KB
data/tweeteval_sentiment_train_labels.txt       91.2 KB
data/tweeteval_sentiment_test_text.txt        1156.9 KB
data/tweeteval_sentiment_test_labels.txt        24.6 KB
data/gsm8k_train.jsonl                        4166.2 KB
data/gsm8k_test.jsonl                          749.7 KB
data/HumanEval.jsonl.gz                         44.9 KB
data/mt_bench_question.jsonl                    48.9 KB

TweetTopic: 6984 tweet, 2019-09-08 s.d. 2021-08-29, 6 topik


File dari GitHub menjadi bahan korpus dan data sentimen. **TweetTopic** memuat 6.984 tweet berlabel enam topik, bertanggal 8 September 2019 sampai 29 Agustus 2021; tanggal itulah yang nanti kita pakai untuk membagi data menurut waktu.

Buku tidak menyertakan korpus pre-training untuk listing-listingnya, jadi kita rakit sendiri korpus mini dari tujuh sumber nyata. Domain setiap dokumen adalah sumbernya.

In [4]:
rows = []
for t in load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train")["text"]:
    t = t.strip()
    if t and not t.startswith("="):  # buang baris kosong dan judul bagian
        rows.append((t, "ensiklopedia (wikitext)"))
for t in open("data/tweeteval_sentiment_train_text.txt", encoding="utf-8"):
    rows.append((t.strip(), "tweet (TweetEval)"))
mmlu = load_dataset("cais/mmlu", "all")
for split in ("test", "validation", "dev"):  # soal ujiannya saja; split auxiliary_train tidak diambil
    for q, choices in zip(mmlu[split]["question"], mmlu[split]["choices"]):
        rows.append((q.strip() + "\n" + "\n".join(choices), "ujian (MMLU)"))
for config in ("ARC-Challenge", "ARC-Easy"):
    arc = load_dataset("allenai/ai2_arc", config)
    for split in arc:
        for q, choices in zip(arc[split]["question"], arc[split]["choices"]):
            rows.append((q.strip() + "\n" + "\n".join(choices["text"]), "sains sekolah (ARC)"))
for path in ("data/gsm8k_train.jsonl", "data/gsm8k_test.jsonl"):
    for line in open(path):
        rows.append((json.loads(line)["question"].strip(), "matematika (GSM8K)"))
for line in gzip.open("data/HumanEval.jsonl.gz", "rt"):
    rows.append((json.loads(line)["prompt"].strip(), "kode (HumanEval)"))
for line in open("data/mt_bench_question.jsonl"):
    for turn in json.loads(line)["turns"]:
        rows.append((turn.strip(), "chat (MT-Bench)"))

CORPUS = pd.DataFrame(rows, columns=["text", "domain"])
CORPUS = CORPUS[CORPUS["text"].str.len() > 0].reset_index(drop=True)
del rows, mmlu, arc
counts = CORPUS["domain"].value_counts()
print(f"korpus pre-training mini: {len(CORPUS)} dokumen\n")
print(pd.DataFrame({"dokumen": counts, "persen": (100 * counts / len(CORPUS)).round(2)}).to_string())

korpus pre-training mini: 95932 dokumen

                         dokumen  persen
domain                                  
tweet (TweetEval)          45615   47.55
ensiklopedia (wikitext)    17556   18.30
ujian (MMLU)               15858   16.53
matematika (GSM8K)          8792    9.16
sains sekolah (ARC)         7787    8.12
kode (HumanEval)             164    0.17
chat (MT-Bench)              160    0.17


Perhatikan bentuk distribusinya. Tweet mengisi 47,55% korpus, sementara kode (164 prompt HumanEval) dan percakapan (160 giliran MT-Bench) masing-masing hanya 0,17%.

**Korpus pre-training sungguhan juga timpang seperti ini**: segelintir sumber besar mendominasi. Domain kecil itulah yang akan menguji listing pertama buku.

## Membagi data pre-training dan fine-tuning

### Stratified sampling untuk data pre-training

Misalkan kita menyisihkan test set acak 10%. Tweet (45.615 dokumen) hampir pasti kebagian sekitar 10%, tetapi bagian kode (164 dokumen) untung-untungan. **Stratified sampling** (pengambilan sampel bertingkat) menghapus untung-untungan itu dengan mengambil 10% dari tiap domain.

> **Listing buku -- nama file contoh**
>
> `your_pretraining_data.csv` hanyalah nama contoh, jadi listing berhenti dengan `FileNotFoundError`. Fungsi `stratified_pretraining_split` sendiri benar; kita pakai tanpa diubah.

In [5]:
# CPU: pembagian data dengan pandas dan scikit-learn, tanpa model.
from sklearn.model_selection import StratifiedShuffleSplit


def stratified_pretraining_split(data, text_column, label_column, test_size=0.1, random_state=42):
    sss = StratifiedShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
    for train_index, test_index in sss.split(data[text_column], data[label_column]):
        train_data = data.iloc[train_index]
        test_data = data.iloc[test_index]
    return train_data, test_data


corpus_train, corpus_test = stratified_pretraining_split(CORPUS, "text", "domain")
print(f"Training set size: {len(corpus_train)}")
print(f"Test set size: {len(corpus_test)}")

Training set size: 86338
Test set size: 9594


Train mendapat 86.338 dokumen dan test 9.594, tepat 90/10. Namun, split acak pun menghasilkan ukuran yang sama. Bedanya ada pada *isi* test set per domain, jadi kita bandingkan dengan 1.000 split acak berukuran sama.

In [6]:
# Berapa dokumen test yang didapat tiap domain: split bertingkat vs 1.000 split acak berukuran sama
domain_of = CORPUS["domain"].to_numpy()
n_test = len(corpus_test)
random_counts = pd.DataFrame([
    pd.Series(domain_of[np.random.default_rng(seed).permutation(len(CORPUS))[:n_test]]).value_counts()
    for seed in range(1000)
]).fillna(0)

table = pd.DataFrame({
    "harapan": (counts * n_test / len(CORPUS)).round(1),
    "bertingkat": corpus_test["domain"].value_counts(),
    "acak_min": random_counts.min().astype(int),
    "acak_maks": random_counts.max().astype(int),
}).loc[counts.index]
print(table.to_string())

                         harapan  bertingkat  acak_min  acak_maks
domain                                                           
tweet (TweetEval)         4561.9        4562      4375       4729
ensiklopedia (wikitext)   1755.7        1756      1640       1877
ujian (MMLU)              1585.9        1586      1476       1687
matematika (GSM8K)         879.3         879       777        965
sains sekolah (ARC)        778.8         779       692        864
kode (HumanEval)            16.4          16         7         30
chat (MT-Bench)             16.0          16         5         29


Kolom `harapan` adalah 10% dari tiap domain. Split bertingkat selalu mendarat di sana; split acak berayun di sekitarnya.

**Untuk domain besar, ayunan itu tak berarti banyak; untuk domain langka, ayunan itu mengubah segalanya.** Tweet mendapat antara 4.375 dan 4.729 dokumen test. Kode mendapat antara 7 dan 30, padahal stratifikasi menjamin 16. Dengan 7 dokumen, satu jawaban saja menggeser akurasi domain itu sekitar 14 poin; dengan 16 dokumen, sekitar 6 poin.

Namun, domain yang terlalu langka justru bisa membuat fungsi buku gagal.

> **Listing buku pada kasus ekstrem -- domain berisi satu dokumen**
>
> Sebuah domain bisa sekecil satu dokumen, misalnya sumber kecil yang hanya menyumbang satu teks. `StratifiedShuffleSplit` lalu berhenti dengan `ValueError: The least populated class in y has only 1 member, which is too few.`, karena satu dokumen tak bisa sekaligus di train dan test. Perbaikannya: masukkan domain sekecil itu seluruhnya ke train, lalu bagi sisanya secara bertingkat.

Kita uji dengan menyisakan satu dokumen di domain kode.

In [7]:
def stratified_split_keep_rare(data, text_column, label_column, min_count=10, **kwargs):
    """Domain dengan dokumen < min_count terlalu kecil untuk dibagi: seluruhnya masuk train, sisanya dibagi bertingkat."""
    small = data[label_column].map(data[label_column].value_counts()) < min_count
    train, test = stratified_pretraining_split(data[~small], text_column, label_column, **kwargs)
    return pd.concat([train, data[small]]), test


# Korpus yang sama, tetapi domain kode tinggal satu dokumen
is_code = CORPUS["domain"] == "kode (HumanEval)"
rare_corpus = pd.concat([CORPUS[~is_code], CORPUS[is_code].head(1)])
train_r, test_r = stratified_split_keep_rare(rare_corpus, "text", "domain")
print(f"train {len(train_r)} | test {len(test_r)}")
print("dokumen kode di train:", (train_r["domain"] == "kode (HumanEval)").sum(),
      "| di test:", (test_r["domain"] == "kode (HumanEval)").sum())

train 86192 | test 9577
dokumen kode di train: 1 | di test: 0


Satu-satunya dokumen kode masuk train, dan hampir 96 ribu dokumen lainnya terbagi normal. **Domain itu tak bisa dievaluasi, tetapi tidak lagi menggagalkan seluruh split.** Ambang `min_count=10` kira-kira setara satu dokumen test pada split 10%: di bawahnya, domain terlalu kecil untuk diukur.

### Time-based splitting untuk data fine-tuning

Bayangkan pengklasifikasi topik yang dilatih pada tweet 2020 lalu dipakai untuk tweet 2021, saat topik dan kosakata sudah bergeser. Split acak menyembunyikan hal ini karena tweet masa depan ikut masuk ke train. Buku menyarankan membagi menurut waktu: masa lalu untuk train, masa depan untuk test.

> **Listing buku -- efek samping tersembunyi**
>
> Selain nama file contoh, baris `data[timestamp_column] = pd.to_datetime(...)` menulis balik ke DataFrame pemanggil: kolom `timestamp` di luar fungsi ikut menjadi `datetime64[ns]`. Versi di bawah memakai variabel lokal.

In [8]:
def time_based_finetuning_split(data, timestamp_column, split_date):
    timestamps = pd.to_datetime(data[timestamp_column])  # perbaikan: kolom milik pemanggil tidak ditimpa
    train_data = data[timestamps < split_date]
    test_data = data[timestamps >= split_date]
    return train_data, test_data


# Urutan baris diacak sekali; split waktu tetap ditentukan tanggal, bukan urutan baris
tweets = (tweet_topic.rename(columns={"date": "timestamp"})[["text", "timestamp", "label", "label_name"]]
          .sample(frac=1, random_state=0)
          .reset_index(drop=True))
split_date = "2020-09-01"  # buku memakai '2023-01-01', tetapi TweetTopic berakhir Agustus 2021
train_data, test_data = time_based_finetuning_split(tweets, "timestamp", split_date)
print(f"Training set size: {len(train_data)}  ({train_data['timestamp'].min()} s.d. {train_data['timestamp'].max()})")
print(f"Test set size: {len(test_data)}  ({test_data['timestamp'].min()} s.d. {test_data['timestamp'].max()})")

Training set size: 3586  (2019-09-08 s.d. 2020-08-31)
Test set size: 3398  (2020-09-04 s.d. 2021-08-29)


Batas 1 September 2020 memberi 3.586 tweet lama untuk train dan 3.398 tweet baru untuk test. Apakah cara membagi ini mengubah penilaian kita? Untuk menjawabnya, kita latih model sekali pada split waktu dan 20 kali pada split acak berukuran sama. Dua puluh satu kali pelatihan butuh pengklasifikasi yang cepat: **TF-IDF**, yang mengubah tiap tweet menjadi bobot kata, lalu regresi logistik di atasnya.

In [9]:
# CPU: TF-IDF dan regresi logistik scikit-learn tidak memakai GPU.
from sklearn import metrics as skm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression


def topic_classifier_scores(train, test):
    """TF-IDF kata 1-2 + regresi logistik, dilatih ulang untuk setiap split. Mengembalikan (akurasi, macro-F1)."""
    vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
    clf = LogisticRegression(max_iter=2000, C=10)
    clf.fit(vectorizer.fit_transform(train["text"]), train["label"])
    pred = clf.predict(vectorizer.transform(test["text"]))
    return skm.accuracy_score(test["label"], pred), skm.f1_score(test["label"], pred, average="macro")


time_acc, time_f1 = topic_classifier_scores(train_data, test_data)
random_scores = []
for seed in range(20):
    perm = np.random.default_rng(seed).permutation(len(tweets))
    random_scores.append(topic_classifier_scores(tweets.iloc[perm[:len(train_data)]], tweets.iloc[perm[len(train_data):]]))
random_scores = np.array(random_scores)

print(f"split waktu    : akurasi {time_acc:.3f} | macro-F1 {time_f1:.3f}")
print(f"20 split acak  : akurasi {random_scores[:, 0].mean():.3f} [{random_scores[:, 0].min():.3f}-{random_scores[:, 0].max():.3f}]"
      f" | macro-F1 {random_scores[:, 1].mean():.3f} [{random_scores[:, 1].min():.3f}-{random_scores[:, 1].max():.3f}]")

split waktu    : akurasi 0.740 | macro-F1 0.435
20 split acak  : akurasi 0.784 [0.769-0.790] | macro-F1 0.562 [0.534-0.579]


Split waktu memberi akurasi 0,740, **lebih rendah daripada semua 20 split acak** (0,769-0,790). Artinya, split acak melaporkan kinerja sekitar 4 poin lebih tinggi daripada yang dialami model pada tweet yang datang kemudian.

Akurasi didominasi topik besar, jadi kita juga melihat **macro-F1**: skor F1 dihitung per topik (seberapa banyak tweet topik itu yang tertangkap, dan seberapa banyak tebakan topik itu yang benar), lalu dirata-ratakan dengan bobot sama. Ukuran ini turun lebih tajam, dari 0,562 ke 0,435, tanda topik kecil kemungkinan besar ikut terpukul. Apa yang berubah antara tweet 2020 dan 2021?

In [10]:
def unseen_word_rate(train, test):
    """Fraksi kata di test (huruf kecil, dipisah spasi) yang tak pernah muncul di train."""
    vocab = {w for text in train["text"] for w in text.lower().split()}
    return np.mean([w not in vocab for text in test["text"] for w in text.lower().split()])


perm = np.random.default_rng(0).permutation(len(tweets))
random_train, random_test = tweets.iloc[perm[:len(train_data)]], tweets.iloc[perm[len(train_data):]]
print(f"kata test yang tak pernah muncul di train: split waktu {unseen_word_rate(train_data, test_data):.1%}"
      f" | split acak {unseen_word_rate(random_train, random_test):.1%}\n")
share = pd.DataFrame({"train (masa lalu)": train_data["label_name"].value_counts(normalize=True),
                      "test (masa depan)": test_data["label_name"].value_counts(normalize=True)})
print((100 * share).round(1).to_string())

kata test yang tak pernah muncul di train: split waktu 18.5% | split acak 16.3%

                          train (masa lalu)  test (masa depan)
label_name                                                    
arts_&_culture                          1.6                2.8
business_&_entrepreneurs                4.2                4.9
daily_life                             16.8               10.5
pop_culture                            38.1               40.0
science_&_technology                    5.3                4.6
sports_&_gaming                        33.9               37.3


Ada dua pergeseran. **Kosakata**: 18,5% kata di test masa depan tak pernah muncul di train, dibanding 16,3% pada split acak. **Komposisi topik**: *daily_life* turun dari 16,8% ke 10,5%, sementara topik terkecil, *arts_&_culture*, naik dari 1,6% ke 2,8%. Topik dengan contoh train paling sedikit kini lebih sering muncul di test, cocok dengan turunnya macro-F1 tadi.

**Split acak menjawab pertanyaan yang berbeda dari yang kita ajukan.** Kita bertanya tentang tweet yang akan datang, tetapi split acak menghapus pergeseran topik dan mengecilkan pergeseran kosakata.

### Oversampling dan pembobotan kelas

Kelas langka sering paling penting, tetapi model mudah mengabaikannya karena jarang dihukum saat salah. Buku menawarkan dua resep: **class weighting** membuat kesalahannya lebih mahal di loss, dan **oversampling** menarik contohnya lebih sering ke dalam batch. Contoh kecil dari buku: tiga label kelas 0, dua kelas 1, dan satu kelas 2.

In [11]:
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler

labels = [0, 0, 0, 1, 1, 2]  # contoh buku: kelas 2 paling jarang
class_weights = compute_class_weight("balanced", classes=np.unique(labels), y=labels)
label_counts = np.bincount(labels)
sample_weights = [1.0 / label_counts[label] for label in labels]
print("bobot kelas  (untuk loss)   :", class_weights.round(3))
print("bobot sampel (untuk sampler):", np.round(sample_weights, 3))

bobot kelas  (untuk loss)   : [0.667 1.    2.   ]
bobot sampel (untuk sampler): [0.333 0.333 0.333 0.5   0.5   1.   ]


Kelas 2 tiga kali lebih jarang daripada kelas 0, jadi kesalahannya tiga kali lebih mahal (2,0 lawan 0,667). Di sampler, tiap contoh kelas 2 tiga kali lebih mungkin ditarik (1,0 lawan 0,333), sehingga ketiga kelas muncul sama sering.

> **Listing buku -- `NameError`: `train_dataset` tidak pernah dibuat**
>
> ```python
> train_loader = DataLoader(train_dataset, sampler=sampler, batch_size=4)
> ```
>
> Bobot di atas benar; listing hanya tak pernah menyiapkan datanya. Data itu kita sediakan di bawah.

Kita uji kedua resep pada **CommitmentBank** (CB). Premis *"Rina bilang mungkin ia datang besok"* tidak memastikan dan tidak membantah hipotesis *"Rina datang besok"*, jadi labelnya *neutral*; bila memastikan, *entailment*; bila membantah, *contradiction*. BERT tidak di-fine-tune: representasinya diambil sekali sebagai **fitur beku**, lalu kita latih **kepala klasifikasi** linear kecil di atasnya.

In [12]:
from transformers import AutoModel, AutoTokenizer

cb = load_dataset("aps/super_glue", "cb")
premises = list(cb["train"]["premise"]) + list(cb["validation"]["premise"])
hypotheses = list(cb["train"]["hypothesis"]) + list(cb["validation"]["hypothesis"])
y_cb = np.array(list(cb["train"]["label"]) + list(cb["validation"]["label"]))
print(f"CommitmentBank: {len(y_cb)} contoh | entailment / contradiction / neutral = {' / '.join(map(str, np.bincount(y_cb)))}")

# Fitur BERT beku: vektor [CLS] digabung rata-rata semua token. Fine-tuning penuh untuk 100 model terlalu mahal, bahkan di GPU.
bert_tok = AutoTokenizer.from_pretrained("bert-base-uncased")
bert_encoder = AutoModel.from_pretrained("bert-base-uncased").to(DEVICE).eval()
encoder_device = model_device(bert_encoder, "BERT encoder")
features = []
with torch.no_grad():
    for start in range(0, len(premises), 16):
        enc = bert_tok(premises[start:start + 16], hypotheses[start:start + 16], truncation=True, padding=True,
                       max_length=256, return_tensors="pt").to(DEVICE)
        hidden = bert_encoder(**enc).last_hidden_state
        mask = enc["attention_mask"].unsqueeze(-1).float()
        mean_pooled = (hidden * mask).sum(1) / mask.sum(1)
        features.append(torch.cat([hidden[:, 0], mean_pooled], dim=-1).cpu())
X_cb = torch.cat(features).numpy()
del bert_encoder, features
free_memory()
print("matriks fitur:", X_cb.shape, "| dihitung di", encoder_device)

CommitmentBank: 306 contoh | entailment / contradiction / neutral = 138 / 147 / 21
matriks fitur: (306, 1536) | dihitung di cuda:0


Dari 306 contoh CB, hanya 21 yang *neutral*: kurang dari 7%. Setiap contoh kini berupa vektor 1.536 angka.

Agar hasilnya tidak bergantung pada satu pembagian yang kebetulan, setiap strategi dinilai dengan **5-fold cross-validation**: data dibagi lima bagian (*fold*), model dilatih lima kali, dan setiap fold sekali menjadi test. Diulang lima kali untuk empat strategi (tanpa koreksi, bobot kelas, oversampling, keduanya), totalnya 100 kepala klasifikasi kecil.

In [13]:
# CPU (sengaja): kepala linear 4.611 parameter terlalu kecil untuk diuntungkan GPU; yang memakan waktu adalah DataLoader.
from sklearn.model_selection import StratifiedKFold


def cb_fit_predict(strategy, train_idx, test_idx, seed, epochs=30):
    """Latih kepala linear di atas fitur beku, kembalikan prediksi untuk test_idx.
    strategy: 'tanpa' | 'bobot kelas' | 'oversampling' | 'keduanya'."""
    torch.manual_seed(seed)
    mu, sd = X_cb[train_idx].mean(0), X_cb[train_idx].std(0) + 1e-6
    X_train = torch.tensor((X_cb[train_idx] - mu) / sd, dtype=torch.float)
    y_train = torch.tensor(y_cb[train_idx])
    head = torch.nn.Linear(X_cb.shape[1], 3)
    optimizer = torch.optim.Adam(head.parameters(), lr=1e-3, weight_decay=1e-3)

    weights = None
    if strategy in ("bobot kelas", "keduanya"):  # resep buku 1: bobot kelas di loss
        weights = torch.tensor(compute_class_weight("balanced", classes=np.unique(y_cb[train_idx]), y=y_cb[train_idx]),
                               dtype=torch.float)
    loss_fn = torch.nn.CrossEntropyLoss(weight=weights)

    generator = torch.Generator().manual_seed(seed)
    if strategy in ("oversampling", "keduanya"):  # resep buku 2: WeightedRandomSampler
        counts_train = np.bincount(y_cb[train_idx])
        sampler = WeightedRandomSampler([1.0 / counts_train[label] for label in y_cb[train_idx]],
                                        num_samples=len(train_idx), replacement=True, generator=generator)
        loader = DataLoader(TensorDataset(X_train, y_train), batch_size=16, sampler=sampler)
    else:
        loader = DataLoader(TensorDataset(X_train, y_train), batch_size=16, shuffle=True, generator=generator)

    for _ in range(epochs):
        for xb, yb in loader:
            optimizer.zero_grad()
            loss_fn(head(xb), yb).backward()
            optimizer.step()
    with torch.no_grad():
        return head(torch.tensor((X_cb[test_idx] - mu) / sd, dtype=torch.float)).argmax(-1).numpy()


rows = []
for strategy in ("tanpa", "bobot kelas", "oversampling", "keduanya"):
    for rep in range(5):  # 5-fold bertingkat, diulang 5 kali dengan pembagian fold berbeda
        pred = np.zeros_like(y_cb)
        folds = StratifiedKFold(5, shuffle=True, random_state=rep).split(X_cb, y_cb)
        for k, (train_idx, test_idx) in enumerate(folds):
            pred[test_idx] = cb_fit_predict(strategy, train_idx, test_idx, seed=10 * rep + k)
        rows.append({"strategi": strategy,
                     "akurasi": (pred == y_cb).mean(),
                     "macro_F1": skm.f1_score(y_cb, pred, average="macro"),
                     "recall_neutral": skm.recall_score(y_cb, pred, labels=[2], average="macro", zero_division=0),
                     "presisi_neutral": skm.precision_score(y_cb, pred, labels=[2], average="macro", zero_division=0),
                     "tebakan_neutral": (pred == 2).sum()})
print(f"rata-rata 5 pengulangan; jumlah contoh neutral sebenarnya: {(y_cb == 2).sum()}")
print(pd.DataFrame(rows).groupby("strategi", sort=False).mean().round(3).to_string())

rata-rata 5 pengulangan; jumlah contoh neutral sebenarnya: 21
              akurasi  macro_F1  recall_neutral  presisi_neutral  tebakan_neutral
strategi                                                                         
tanpa           0.776     0.667           0.400            0.381             22.2
bobot kelas     0.771     0.666           0.448            0.350             27.2
oversampling    0.773     0.669           0.457            0.351             27.8
keduanya        0.758     0.662           0.543            0.308             37.4


Tanpa koreksi, model sudah menebak *neutral* rata-rata 22,2 kali, hampir sama dengan 21 contoh yang ada. Masalahnya tebakan itu sering meleset: **recall** 0,400 berarti sekitar 8 dari 21 contoh *neutral* tertangkap, dan **presisi** 0,381 berarti sekitar 8 dari 22 tebakan *neutral* benar.

Bobot kelas *atau* oversampling menaikkan recall ke sekitar 0,45, dengan harga presisi turun. **Memakai keduanya mengoreksi ketimpangan yang sama dua kali**: tebakan *neutral* melonjak ke 37,4, dengan presisi terendah (0,308). Macro-F1 nyaris diam (0,662-0,669): koreksi ini memindahkan kesalahan, bukan menguranginya. Apa yang sebenarnya ditarik sampler dalam satu epoch?

In [14]:
# Satu epoch WeightedRandomSampler atas 306 contoh: apa yang sebenarnya dilihat model?
counts_cb = np.bincount(y_cb)
sampler = WeightedRandomSampler([1.0 / counts_cb[label] for label in y_cb], num_samples=len(y_cb), replacement=True,
                                generator=torch.Generator().manual_seed(0))
drawn = np.array(list(sampler))
seen = np.unique(drawn)
print("ditarik per kelas      :", np.bincount(y_cb[drawn], minlength=3))
print("contoh unik per kelas  :", np.bincount(y_cb[seen], minlength=3), "dari", counts_cb)
print(f"contoh unik total      : {len(seen)} dari {len(y_cb)}")

ditarik per kelas      : [100 105 101]
contoh unik per kelas  : [70 74 21] dari [138 147  21]
contoh unik total      : 165 dari 306


Satu epoch oversampling menarik 100/105/101 contoh per kelas: seimbang, seperti dijanjikan. Namun, hanya 165 dari 306 contoh yang muncul. Semua 21 contoh *neutral* terlihat, rata-rata hampir lima kali, sementara 68 *entailment* dan 73 *contradiction* tidak tersentuh.

**Oversampling tidak menghapus ketimpangan, tetapi mengubahnya menjadi pengulangan.** Mengulang 21 contoh yang sama mudah berujung overfitting.

### Under the hood (UTH-1): satu split atau k-fold?

Di bagian CB tadi kita memakai 5-fold tanpa mempersoalkannya. Bab ini dibuka dengan klaim bahwa cross-validation lebih kokoh daripada satu split. Apakah lima kali pelatihan itu sepadan? Dengan data dan model yang sama, kita bandingkan sebaran akurasi 200 split tunggal 80/20 dengan 40 pengulangan **k-fold cross-validation** (*k* = 5).

In [15]:
# CPU (sengaja), sama seperti sel strategi di atas: 400 kepala linear kecil, masing-masing dilatih ulang dari nol.
single_split = []
for seed in range(200):  # 200 split tunggal 80/20, masing-masing satu kali latih
    train_idx, test_idx = next(StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=seed).split(X_cb, y_cb))
    single_split.append((cb_fit_predict("tanpa", train_idx, test_idx, seed=seed) == y_cb[test_idx]).mean())
print("200 split tunggal selesai; lanjut 40 pengulangan 5-fold")

kfold = []
for rep in range(40):  # 40 pengulangan 5-fold, masing-masing lima kali latih
    pred = np.zeros_like(y_cb)
    folds = StratifiedKFold(5, shuffle=True, random_state=1000 + rep).split(X_cb, y_cb)
    for k, (train_idx, test_idx) in enumerate(folds):
        pred[test_idx] = cb_fit_predict("tanpa", train_idx, test_idx, seed=1000 + 10 * rep + k)
    kfold.append((pred == y_cb).mean())

for name, acc, fits in (("split tunggal 80/20", np.array(single_split), 1), ("5-fold CV", np.array(kfold), 5)):
    print(f"{name:<20} latih per estimasi: {fits} | rata-rata {acc.mean():.3f} | sd {acc.std():.3f}"
          f" | min-maks {acc.min():.3f}-{acc.max():.3f}")
print(f"varians split tunggal / varians 5-fold: {np.var(single_split) / np.var(kfold):.0f}x")

200 split tunggal selesai; lanjut 40 pengulangan 5-fold
split tunggal 80/20  latih per estimasi: 1 | rata-rata 0.782 | sd 0.042 | min-maks 0.677-0.903
5-fold CV            latih per estimasi: 5 | rata-rata 0.778 | sd 0.013 | min-maks 0.745-0.810
varians split tunggal / varians 5-fold: 11x


**Hasil satu split sangat bergantung pada keberuntungan.** Dua ratus split 80/20 memberi akurasi 0,677 sampai 0,903, padahal yang berubah hanya cara membagi 306 contoh yang sama dan seed latihnya. Empat puluh pengulangan 5-fold berkumpul di 0,745 sampai 0,810: varians 11 kali lebih kecil, dengan harga lima kali pelatihan.

Laporkan rata-rata dan simpangan baku antarpengulangan. Ingat saja bahwa simpangan baku itu hanya mengukur ayunan akibat cara membagi dan seed latih, bukan ketidakpastian karena datanya cuma 306 contoh.

## Evaluasi few-shot dan zero-shot

Sejauh ini kita membagi data untuk melatih model. LLM juga bisa dipakai tanpa dilatih ulang, cukup lewat prompt. Tabel 15.1 buku membandingkan dua caranya: **zero-shot** hanya memberi instruksi, **few-shot** menambahkan beberapa contoh berlabel. Menurut buku, few-shot biasanya lebih akurat, tetapi contohnya harus dipilih hati-hati. Kedua klaim ini akan kita uji.

### Evaluasi few-shot

Listing buku memberi dua ulasan berlabel, lalu meminta model melanjutkan teks setelah `Output:` untuk ulasan ketiga. Kita pakai model yang sama dengan buku, `gpt2-large`.

In [16]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer


def few_shot_evaluate(model, tokenizer, task_description, examples, test_instance):
    prompt = f"{task_description}\n\nExamples:\n"
    for example in examples:
        prompt += (
            f"Input: {example['input']}\n"
            f"Output: {example['output']}\n\n"
        )
    prompt += f"Input: {test_instance}\nOutput:"
    input_ids = tokenizer.encode(prompt, return_tensors='pt').to(model.device)  # edisi GPU: input ikut ke perangkat model
    with torch.no_grad():
        output = model.generate(input_ids, max_length=100, num_return_sequences=1)
    generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
    return generated_text.split("Output:")[-1].strip()


# Model buku: gpt2-large (774M parameter, unduhan sekitar 3,2 GB, sekitar 3 GB VRAM dalam FP32).
model = GPT2LMHeadModel.from_pretrained('gpt2-large').to(DEVICE)  # edisi GPU
tokenizer = GPT2Tokenizer.from_pretrained('gpt2-large')
print("gpt2-large berjalan di", model_device(model, "gpt2-large"))
task_description = "Classify the sentiment of the following movie reviews as positive or negative."
examples = [
    {"input": "This movie was fantastic!", "output": "Positive"},
    {"input": "I hated every minute of it.", "output": "Negative"}
]
test_instance = "The acting was superb, but the plot was confusing."
result = few_shot_evaluate(model, tokenizer, task_description, examples, test_instance)
print(f"Few-shot evaluation result: {result}")

gpt2-large berjalan di cuda:0
Few-shot evaluation result: Positive

Input: The acting was great, but the plot was boring.


Model menjawab *Positive* untuk ulasan campuran itu (*superb* tapi *confusing*). Namun, ada yang janggal: output ikut memuat baris `Input:` berisi ulasan yang tidak pernah kita tulis. Untuk melacak asalnya, kita jalankan prompt yang sama dan cetak seluruh lanjutan model.

In [17]:
# Prompt dan generate yang sama dengan fungsi buku; kali ini lanjutan model dicetak utuh
prompt = f"{task_description}\n\nExamples:\n"
prompt += "".join(f"Input: {e['input']}\nOutput: {e['output']}\n\n" for e in examples)
prompt += f"Input: {test_instance}\nOutput:"
input_ids = tokenizer.encode(prompt, return_tensors="pt").to(model.device)
with torch.no_grad():
    output = model.generate(input_ids, max_length=100, num_return_sequences=1)
continuation = tokenizer.decode(output[0][input_ids.shape[1]:].cpu(), skip_special_tokens=True)
print(f"token prompt {input_ids.shape[1]} + token baru {output.shape[1] - input_ids.shape[1]} = {output.shape[1]} (max_length=100)")
print("lanjutan model:", repr(continuation))
print("'Output:' di prompt:", prompt.count("Output:"), "| di teks lengkap:", (prompt + continuation).count("Output:"))

token prompt 63 + token baru 37 = 100 (max_length=100)
lanjutan model: ' Positive\n\nInput: The story was interesting, but the characters were flat.\n\nOutput: Positive\n\nInput: The acting was great, but the plot was boring.\n'
'Output:' di prompt: 3 | di teks lengkap: 4


GPT-2 tidak berhenti setelah menjawab. Ia meneruskan pola prompt: mengarang ulasan baru, memberinya `Output: Positive`, lalu memulai ulasan berikutnya sampai jatah `max_length=100` habis (63 token di antaranya milik prompt). Teks lengkap kini memuat empat `Output:`, padahal prompt hanya tiga.

**Fungsi buku mengambil jawaban dari tempat yang salah.** `split("Output:")[-1]` memotong di `Output:` terakhir, milik ulasan karangan model. Hasilnya kebetulan tetap *Positive*, karena ulasan karangan itu juga diberi label *Positive*. Jawaban model untuk ulasan kita sebenarnya kata pertama lanjutannya, tepat setelah `Output:` ketiga.

Lagi pula, satu ulasan tanpa label acuan belum bisa disebut evaluasi. Itu kita kerjakan setelah melihat zero-shot.

### Evaluasi zero-shot

Sekarang semua contoh kita buang. Listing buku meminta model memilih satu dari empat kategori untuk kalimat tentang penjelajah Mars milik NASA.

In [18]:
def zero_shot_evaluate(model, tokenizer, task_description, test_instance):
    prompt = f"{task_description}\n\nInput: {test_instance}\nOutput:"
    input_ids = tokenizer.encode(prompt, return_tensors='pt').to(model.device)  # edisi GPU
    with torch.no_grad():
        output = model.generate(input_ids, max_length=100, num_return_sequences=1)
    generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
    return generated_text.split("Output:")[-1].strip()


task_description = "Classify the following text into one of these categories: Science, Politics, Sports, Entertainment."
test_instance = "NASA's Mars rover has discovered traces of ancient microbial life."
result = zero_shot_evaluate(model, tokenizer, task_description, test_instance)
print(f"Zero-shot evaluation result: {result}")

Zero-shot evaluation result: A new book by the author of "The Martian" and "The Martian Chronicles" about the search for life on Mars.

The Martian

By Andy Weir

Hardcover, 304 pages

HarperCollins

List price: $29.95

The Martian is a novel


Menurut buku, model akan menyimpulkan kategori dari pengetahuannya. Yang keluar justru mirip katalog toko buku (novel *The Martian*, penulis, jumlah halaman, harga), bukan salah satu kategori.

**Generasi bebas tidak mengklasifikasikan.** Model hanya melanjutkan teks, tanpa dipaksa memilih label. Klasifikasi seharusnya bertanya langsung: berapa peluang token berikutnya `Science`, berapa `Politics`, dan seterusnya, lalu memilih yang tertinggi. Fungsi `label_scores` di bawah melakukannya dengan membaca log-probabilitas token label di posisi terakhir prompt.

In [19]:
from transformers import GPT2TokenizerFast

gpt2_tok = GPT2TokenizerFast.from_pretrained("gpt2")  # kosakata gpt2 dan gpt2-large identik
gpt2 = GPT2LMHeadModel.from_pretrained("gpt2").to(DEVICE).eval()
print("GPT-2 124M berjalan di", model_device(gpt2, "GPT-2"))


@torch.no_grad()
def label_scores(prefix, suffixes, label_ids, model=None, batch_size=32):
    """log p(token label | prefix + suffix), dibaca di posisi token terakhir.
    Prefix (instruksi + contoh) diproses sekali; cache key/value-nya dipakai ulang untuk setiap suffix."""
    model = model or gpt2
    dev = model.device
    prefix_ids = gpt2_tok.encode(prefix) if prefix else []
    if prefix_ids:
        assert gpt2_tok.encode(prefix + suffixes[0]) == prefix_ids + gpt2_tok.encode(suffixes[0]), "batas token prefix tidak stabil"
        prefix_cache = model.transformer(input_ids=torch.tensor([prefix_ids], device=dev), use_cache=True).past_key_values
    scores = []
    for start in range(0, len(suffixes), batch_size):
        batch = [gpt2_tok.encode(s) for s in suffixes[start:start + batch_size]]
        n, width = len(batch), max(map(len, batch))
        input_ids = torch.full((n, width), gpt2_tok.eos_token_id)
        attention = torch.zeros((n, len(prefix_ids) + width), dtype=torch.long)
        attention[:, :len(prefix_ids)] = 1
        for row, ids in enumerate(batch):
            input_ids[row, :len(ids)] = torch.tensor(ids)
            attention[row, len(prefix_ids):len(prefix_ids) + len(ids)] = 1
        kwargs = {"use_cache": False}
        if prefix_ids:
            cache = copy.deepcopy(prefix_cache)
            cache.batch_repeat_interleave(n)
            kwargs = {"past_key_values": cache, "use_cache": True}
        hidden = model.transformer(input_ids=input_ids.to(dev), attention_mask=attention.to(dev), **kwargs).last_hidden_state
        last = torch.tensor([len(ids) - 1 for ids in batch], device=dev)
        logits = model.lm_head(hidden[torch.arange(n, device=dev), last]).float()
        scores.append(F.log_softmax(logits, dim=-1)[:, label_ids].cpu())
    return torch.cat(scores).numpy()


# Kalimat NASA yang sama, sekarang sebagai pilihan di antara empat label
categories = ["Science", "Politics", "Sports", "Entertainment"]
category_ids = [gpt2_tok.encode(" " + c)[0] for c in categories]
for name, lm in (("gpt2-large (model buku)", model), ("gpt2 (124M)", gpt2)):
    s = label_scores(f"{task_description}\n\nInput:", [f" {test_instance}\nOutput:"], category_ids, model=lm)[0]
    probs = np.exp(s - np.logaddexp.reduce(s))
    print(f"{name:<24} " + " | ".join(f"{c} {p:.2f}" for c, p in zip(categories, probs)))
del model, tokenizer  # gpt2-large (sekitar 3 GB) dilepas; sisa bab memakai gpt2 (124M)
free_memory()
print("\n" + memory_report())

GPT-2 124M berjalan di cuda:0
gpt2-large (model buku)  Science 0.95 | Politics 0.02 | Sports 0.02 | Entertainment 0.01
gpt2 (124M)              Science 0.99 | Politics 0.00 | Sports 0.00 | Entertainment 0.00

VRAM terpakai 0.51 GB (puncak sejak laporan sebelumnya 3.71 GB)


Ditanya dengan cara ini, kedua model memilih *Science* dengan yakin: peluangnya di antara empat label 0,95 untuk `gpt2-large` dan 0,99 untuk GPT-2 kecil (124M). **Untuk kalimat ini, pengetahuannya ada; yang salah cara bertanyanya.**

GPT-2 124M pun menjawab benar, padahal ukurannya hanya sekitar seperenam `gpt2-large`, jadi mulai dari sini kita memakai model kecil itu. Baris terakhir output memastikan `gpt2-large` sudah dilepas: VRAM yang masih terpakai mestinya sekitar 0,5 GB, yaitu GPT-2 124M saja, sedangkan angka puncaknya mencatat saat `gpt2-large` masih dimuat.

Apakah few-shot memang lebih akurat daripada zero-shot? Kita ukur pada 872 ulasan film berlabel dari **SST-2**, dengan format prompt buku.

In [20]:
def boot_ci(x, n_boot=2000, seed=0):
    """Rata-rata beserta interval bootstrap 95%."""
    x = np.asarray(x, dtype=float)
    means = x[np.random.default_rng(seed).integers(0, len(x), (n_boot, len(x)))].mean(1)
    return x.mean(), np.percentile(means, 2.5), np.percentile(means, 97.5)


sst_val = load_dataset("nyu-mll/glue", "sst2", split="validation")
sst_train_full = load_dataset("nyu-mll/glue", "sst2", split="train")  # dipakai lagi untuk contoh few-shot dan fine-tuning
NAME = ["Negative", "Positive"]
SX = [x.strip() for x in sst_val["sentence"]]
SY = np.array(sst_val["label"])
TASK = "Classify the sentiment of the following movie reviews as positive or negative."
SUFFIXES = [f" {x}\nOutput:" for x in SX]
LABEL_IDS = [gpt2_tok.encode(" Negative")[0], gpt2_tok.encode(" Positive")[0]]


def fs_prefix(demos):
    """Format prompt buku, dipotong tepat setelah 'Input:' terakhir agar batas tokennya stabil untuk cache."""
    if not demos:
        return f"{TASK}\n\nInput:"
    shots = "".join(f"Input: {text}\nOutput: {label}\n\n" for text, label in demos)
    return f"{TASK}\n\nExamples:\n{shots}Input:"


def sst_eval(demos):
    s = label_scores(fs_prefix(demos), SUFFIXES, LABEL_IDS)
    pred = (s[:, 1] > s[:, 0]).astype(int)
    return (pred == SY).astype(float), pred.mean(), s


print(f"SST-2 validation: {len(SY)} kalimat, {SY.mean():.1%} positif -> baseline kelas mayoritas {max(SY.mean(), 1 - SY.mean()):.3f}")
BOOK_DEMOS = [("This movie was fantastic!", "Positive"), ("I hated every minute of it.", "Negative")]
for name, demos in (("zero-shot", []), ("few-shot, 2 contoh buku", BOOK_DEMOS)):
    hits, positive_rate, _ = sst_eval(demos)
    mean, lo, hi = boot_ci(hits)
    print(f"{name:<24} akurasi {mean:.3f} [CI95 {lo:.3f}-{hi:.3f}] | ditebak positif {positive_rate:.1%}")

SST-2 validation: 872 kalimat, 50.9% positif -> baseline kelas mayoritas 0.509
zero-shot                akurasi 0.694 [CI95 0.663-0.724] | ditebak positif 55.4%
few-shot, 2 contoh buku  akurasi 0.678 [CI95 0.647-0.708] | ditebak positif 36.1%


Zero-shot meraih 0,694, jauh di atas **baseline kelas mayoritas** 0,509 (selalu menebak label terbanyak). Dalam kurung siku ada **interval bootstrap 95%**: rentang yang masuk akal bagi akurasi *sebenarnya* model, mengingat kita baru menguji 872 kalimat.

Few-shot dengan dua contoh buku meraih 0,679, masih di dalam interval itu: contohnya tidak menambah akurasi. Yang berubah adalah tebakan positif, dari 55,4% (dekat porsi sebenarnya, 50,9%) ke 36,0%. **Contoh few-shot ikut menggeser model ke salah satu label.**

Bagaimana bila labelnya lebih dari dua? Kita coba 600 tweet TweetTopic, dengan satu kata label per topik.

In [21]:
TOPIC_WORDS = [" arts", " business", " pop", " life", " sports", " science"]  # satu token per topik, urut id label
TOPIC_TASK = "Classify the following tweet into one of these topics: arts, business, pop, life, sports, science."
tt_test = pd.read_json(TT_FILES["test_2021"], lines=True, convert_dates=False)
tt_test = tt_test.sample(600, random_state=0)  # 600 dari 1.693 tweet, sampel tetap (random_state=0)
s = label_scores(f"{TOPIC_TASK}\n\nInput:", [f" {t}\nOutput:" for t in tt_test["text"]],
                 [gpt2_tok.encode(w)[0] for w in TOPIC_WORDS])
pred, truth = s.argmax(1), tt_test["label"].to_numpy()
print(f"TweetTopic test 2021, {len(tt_test)} tweet: akurasi zero-shot {(pred == truth).mean():.3f}"
      f" | macro-F1 {skm.f1_score(truth, pred, average='macro'):.3f}")
majority_share = tt_test["label_name"].value_counts(normalize=True)
print(f"baseline mayoritas ({majority_share.index[0]}): {majority_share.iloc[0]:.3f}")
print("tebakan per topik:", dict(zip([w.strip() for w in TOPIC_WORDS], np.bincount(pred, minlength=6).tolist())))

TweetTopic test 2021, 600 tweet: akurasi zero-shot 0.502 | macro-F1 0.395
baseline mayoritas (sports_&_gaming): 0.397
tebakan per topik: {'arts': 34, 'business': 40, 'pop': 51, 'life': 138, 'sports': 321, 'science': 16}


Pada TweetTopic, zero-shot meraih akurasi 0,502, di atas baseline mayoritas 0,397. Namun, tebakannya timpang: 321 dari 600 tweet ditebak *sports*, sedangkan *pop* hanya 51 kali, padahal *pop_culture* termasuk topik terbesar. Satu tersangka adalah kata labelnya: `pop` juga bisa berarti musik pop atau bunyi letusan. **Kata label adalah bagian dari prompt, dan prompt adalah bagian dari metrik.**

### Under the hood (UTH-2): satu angka few-shot hanyalah satu sampel

Kembali ke SST-2. Dua contoh buku tadi menggeser tebakan positif dari 55,4% ke 36,0%. Bagaimana bila contohnya diganti, atau urutannya saja yang diubah? Kita coba 14 prompt berisi empat contoh dari train SST-2: lima set acak, lima set seimbang, dan empat urutan dari satu set.

**Kalibrasi kontekstual** (Zhao dkk., 2021) ikut diuji: skor tiap label dikurangi skornya untuk input tanpa isi (`N/A`), untuk menetralkan pergeseran seperti tadi.

In [22]:
trX = [x.strip() for x in sst_train_full["sentence"]]
trY = np.array(sst_train_full["label"])
# Contoh few-shot hanya dari kalimat >= 6 kata, karena train SST-2 juga berisi potongan frasa
pool = np.array([i for i, x in enumerate(trX) if len(x.split()) >= 6])


def run_prompt(demo_idx):
    """Akurasi few-shot, akurasi setelah kalibrasi kontekstual, dan proporsi tebakan positif."""
    demos = [(trX[i], NAME[trY[i]]) for i in demo_idx]
    hits, positive_rate, s = sst_eval(demos)
    content_free = label_scores(fs_prefix(demos), [" N/A\nOutput:"], LABEL_IDS)[0]  # Zhao dkk. 2021
    calibrated = s - content_free
    calibrated_acc = ((calibrated[:, 1] > calibrated[:, 0]).astype(int) == SY).mean()
    return hits.mean(), calibrated_acc, positive_rate


configs = []
for seed in range(5):  # A: empat contoh acak
    configs.append(("acak", np.random.default_rng(seed).choice(pool, 4, replace=False)))
balanced = []
for seed in range(5):  # B: dua positif + dua negatif, urutan diacak
    rng = np.random.default_rng(100 + seed)
    idx = np.concatenate([rng.choice(pool[trY[pool] == c], 2, replace=False) for c in (0, 1)])
    rng.shuffle(idx)
    balanced.append(idx)
    configs.append(("seimbang", idx))
pos_i = [i for i in balanced[0] if trY[i] == 1]  # C: himpunan seimbang pertama, empat urutan
neg_i = [i for i in balanced[0] if trY[i] == 0]
for order in ([*pos_i, *neg_i], [*neg_i, *pos_i], [pos_i[0], neg_i[0], pos_i[1], neg_i[1]], [neg_i[0], pos_i[0], neg_i[1], pos_i[1]]):
    configs.append(("urutan", np.array(order)))

rows = []
for group, idx in configs:
    acc, cal, pos = run_prompt(idx)
    rows.append({"kelompok": group, "label_contoh": "".join(NAME[trY[i]][0] for i in idx),
                 "akurasi": acc, "terkalibrasi": cal, "positif_%": 100 * pos})
R = pd.DataFrame(rows)
print(f"SST-2 validation, k=4, GPT-2 124M, {len(R)} prompt\n")
print(R.round({"akurasi": 3, "terkalibrasi": 3, "positif_%": 1}).to_string(index=False))
print("\n" + R.groupby("kelompok", sort=False)["akurasi"].agg(["mean", "std", "min", "max"]).round(3).to_string())

SST-2 validation, k=4, GPT-2 124M, 14 prompt

kelompok label_contoh  akurasi  terkalibrasi  positif_%
    acak         PPPP    0.514         0.510       99.5
    acak         PPPN    0.714         0.513       65.5
    acak         PNPP    0.587         0.571       13.3
    acak         NNPN    0.683         0.544       69.5
    acak         PPPP    0.514         0.509       99.5
seimbang         PNNP    0.517         0.586        5.2
seimbang         PNNP    0.508         0.632        2.2
seimbang         PPNN    0.681         0.524       69.7
seimbang         PNNP    0.513         0.558        3.6
seimbang         NPPN    0.515         0.538        9.3
  urutan         PPNN    0.519         0.517       98.5
  urutan         NNPP    0.701         0.545       40.9
  urutan         PNPN    0.524         0.513       95.3
  urutan         NPNP    0.542         0.622        9.1

           mean    std    min    max
kelompok                            
acak      0.603  0.094  0.514  0.714
se

Empat belas prompt yang sah menurut Tabel 15.1 menghasilkan akurasi **0,508 sampai 0,714**; zero-shot (0,694) mengalahkan 12 di antaranya.

Kolom `positif_%` menunjukkan sebabnya: sembilan prompt membuat model menebak positif untuk lebih dari 95% kalimat, atau kurang dari 10%. Model **runtuh ke satu label**. Contoh seimbang tidak menolong: rata-ratanya 0,547, malah di bawah set acak (0,603). Urutan saja mengayunkan akurasi dari 0,519 ke 0,701. Kalibrasi kontekstual hanya menolong prompt yang runtuh ke *negative*, dan menurunkan yang terbaik (0,714 menjadi 0,513).

**Pada GPT-2 124M, satu angka few-shot hanyalah satu sampel.** Laporkan sebarannya atas beberapa set contoh dan urutan, seperti k-fold atas fold-nya.

## Generalisasi domain dan tugas

### Evaluasi adaptasi domain

Bayangkan model sentimen yang dilatih pada ulasan film (**domain sumber**) lalu dipakai untuk tweet (**domain target**). Buku mengukur dampaknya sebagai ***adaptation drop***: akurasi sumber dikurangi akurasi target.

> **Listing buku -- tiga penyebab gagal**
>
> ```python
> source_domain_data = load_source_domain_data()
> outputs = model(inputs)
> ```
>
> Pemuat data tak pernah ditulis dan `accuracy_score` tak di-import: dua `NameError`. `model(inputs)` menyerahkan seluruh hasil tokenizer sebagai satu argumen; BERT menganggapnya `input_ids` dan gagal dengan `AttributeError`. Seharusnya `model(**inputs)`.

Pertama, model domain sumber: BERT di-fine-tune pada 1.000 kalimat dan frasa ulasan dari train SST-2.

In [23]:
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments
from transformers.trainer_callback import PrinterCallback


def finetune_bert(texts, labels, seed=0):
    """BERT + kepala klasifikasi dua kelas, satu epoch. Di bab ini selalu 1.000 contoh, agar model SST-2 dan model tweet
    mendapat anggaran yang sama."""
    transformers.set_seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)
    data = datasets.Dataset.from_dict({"text": texts, "label": labels}).map(
        lambda batch: bert_tok(batch["text"], truncation=True, max_length=128), batched=True, remove_columns=["text"])
    args = TrainingArguments(output_dir="./results_da", num_train_epochs=1, learning_rate=2e-5,
                             per_device_train_batch_size=16, save_strategy="no", logging_strategy="no",
                             disable_tqdm=True, report_to="none", use_cpu=USE_CPU,
                             dataloader_pin_memory=DEVICE.type == "cuda", seed=seed)
    trainer = Trainer(model=model, args=args, train_dataset=data, processing_class=bert_tok)
    trainer.remove_callback(PrinterCallback)
    trainer.train()
    model_device(model, "BERT hasil fine-tuning")  # Trainer harus memakai perangkat yang diminta
    del trainer  # optimizer dan state Trainer dilepas sebelum model berikutnya
    free_memory()
    return model.eval()


sst_train_1k = sst_train_full.shuffle(seed=0).select(range(1000))
model_sst = finetune_bert([x.strip() for x in sst_train_1k["sentence"]], list(sst_train_1k["label"]))
print(f"BERT di-fine-tune pada {len(sst_train_1k)} kalimat SST-2, 1 epoch, di {model_device(model_sst, 'BERT')}")

BERT di-fine-tune pada 1000 kalimat SST-2, 1 epoch, di cuda:0


Model domain sumber sudah siap. Sekarang kita ukur akurasinya dengan fungsi buku yang sudah diperbaiki: sumbernya 872 ulasan film dari validation SST-2, targetnya 1.000 tweet dari TweetEval. Tweet netral dibuang supaya labelnya sama dengan SST-2: positif atau negatif.

In [24]:
from sklearn.metrics import accuracy_score  # perbaikan 1: import yang tidak ditulis buku


def evaluate_domain_adaptation(model, tokenizer, source_domain_data, target_domain_data):
    def predict(text):
        inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True).to(model.device)
        with torch.no_grad():
            outputs = model(**inputs)  # perbaikan 2: buku menulis model(inputs)
        return torch.argmax(outputs.logits, dim=1).item()

    source_predictions = [predict(text) for text in source_domain_data["text"]]
    source_accuracy = accuracy_score(source_domain_data["label"], source_predictions)
    target_predictions = [predict(text) for text in target_domain_data["text"]]
    target_accuracy = accuracy_score(target_domain_data["label"], target_predictions)
    return {"source_accuracy": source_accuracy,
            "target_accuracy": target_accuracy,
            "adaptation_drop": source_accuracy - target_accuracy,
            "predictions": (np.array(source_predictions), np.array(target_predictions))}  # tambahan untuk analisis


def load_tweeteval(split):
    """TweetEval sentimen dibuat biner seperti SST-2: tweet netral dibuang, positif = 1."""
    texts = [line.rstrip("\n") for line in open(f"data/tweeteval_sentiment_{split}_text.txt", encoding="utf-8")]
    labels = [int(line) for line in open(f"data/tweeteval_sentiment_{split}_labels.txt")]
    frame = pd.DataFrame({"text": texts, "label": labels})
    return frame[frame["label"] != 1].assign(label=lambda d: (d["label"] == 2).astype(int))


source_domain_data = pd.DataFrame({"text": SX, "label": SY})  # sumber: ulasan film (SST-2 validation)
target_domain_data = load_tweeteval("test").sample(1000, random_state=0).reset_index(drop=True)  # target: 1.000 tweet
results = evaluate_domain_adaptation(model_sst, bert_tok, source_domain_data, target_domain_data)
print(f"Source domain accuracy: {results['source_accuracy']:.2f}")
print(f"Target domain accuracy: {results['target_accuracy']:.2f}")
print(f"Adaptation drop: {results['adaptation_drop']:.2f}")
del model_sst  # model SST-2 tidak dipakai lagi; prediksinya tersimpan di results
free_memory()

Source domain accuracy: 0.83
Target domain accuracy: 0.78
Adaptation drop: 0.05


Angkanya tampak wajar: run acuan mencatat akurasi 0,84 di ulasan film dan 0,74 di tweet, jadi *adaptation drop* 0,10. Fine-tuning di GPU bisa memberi angka lain, tetapi cara memeriksanya sama: sebelum percaya, kita letakkan angka itu di samping baseline kelas mayoritas tiap domain.

In [25]:
src_pred, tgt_pred = results["predictions"]
tgt_y = target_domain_data["label"].to_numpy()
src_hit, tgt_hit = (src_pred == SY).astype(float), (tgt_pred == tgt_y).astype(float)
rng = np.random.default_rng(0)
drops = []
for _ in range(2000):  # bootstrap selisih akurasi: sampel ulang kedua domain secara terpisah
    drops.append(src_hit[rng.integers(0, len(src_hit), len(src_hit))].mean()
                 - tgt_hit[rng.integers(0, len(tgt_hit), len(tgt_hit))].mean())

print(f"{'':<22}{'sumber (ulasan)':>17}{'target (tweet)':>16}")
print(f"{'baseline mayoritas':<22}{max(SY.mean(), 1 - SY.mean()):>17.3f}{max(tgt_y.mean(), 1 - tgt_y.mean()):>16.3f}")
print(f"{'akurasi':<22}{src_hit.mean():>17.3f}{tgt_hit.mean():>16.3f}")
balanced_src = skm.balanced_accuracy_score(SY, src_pred)
balanced_tgt = skm.balanced_accuracy_score(tgt_y, tgt_pred)
print(f"{'balanced accuracy':<22}{balanced_src:>17.3f}{balanced_tgt:>16.3f}")
print(f"{'ditebak positif':<22}{src_pred.mean():>17.1%}{tgt_pred.mean():>16.1%}")
print(f"{'positif sebenarnya':<22}{SY.mean():>17.1%}{tgt_y.mean():>16.1%}")
low, high = np.percentile(drops, [2.5, 97.5])
print(f"\npenurunan akurasi {src_hit.mean() - tgt_hit.mean():.3f} [CI95 {low:.3f}-{high:.3f}]")

                        sumber (ulasan)  target (tweet)
baseline mayoritas                0.509           0.637
akurasi                           0.831           0.783
balanced accuracy                 0.832           0.717
ditebak positif                   49.0%           20.0%
positif sebenarnya                50.9%           36.3%

penurunan akurasi 0.048 [CI95 0.012-0.084]


**Selisih akurasi dua domain bisa menyesatkan bila proporsi kelasnya berbeda.** Ulasan film hampir seimbang, sedangkan 63,7% tweet negatif. Model hanya menebak positif untuk 14,1% tweet, searah dengan mayoritas itu, sehingga akurasinya terdongkrak: 0,742 hanya 10,5 poin di atas tebakan "selalu negatif".

**Balanced accuracy**, rata-rata recall tiap kelas, kebal terhadap proporsi itu, dan turun 18,4 poin: hampir dua kali penurunan akurasi (9,6 poin).

Bagaimana bila BERT dilatih langsung pada tweet? Kita pakai anggaran yang sama (1.000 contoh, satu epoch) dengan tiga seed.

In [26]:
@torch.no_grad()
def predict_batch(model, texts, batch_size=64):
    """Sama dengan predict() di atas, tetapi 64 kalimat per forward agar cepat."""
    preds = []
    for start in range(0, len(texts), batch_size):
        enc = bert_tok(list(texts[start:start + batch_size]), return_tensors="pt", truncation=True,
                       padding=True).to(model.device)
        preds.append(model(**enc).logits.argmax(-1).cpu())
    return torch.cat(preds).numpy()


# Model "plafon": anggaran sama (1.000 contoh, 1 epoch), tetapi dilatih langsung pada tweet, dengan tiga seed
tweet_train_1k = load_tweeteval("train").sample(1000, random_state=0)
rows = []
for seed in (0, 1, 2):
    model_tw = finetune_bert(list(tweet_train_1k["text"]), list(tweet_train_1k["label"]), seed=seed)
    pred_review, pred_tweet = predict_batch(model_tw, SX), predict_batch(model_tw, target_domain_data["text"])
    rows.append({"seed": seed,
                 "akurasi_tweet": (pred_tweet == tgt_y).mean(),
                 "balanced_tweet": skm.balanced_accuracy_score(tgt_y, pred_tweet),
                 "balanced_ulasan": skm.balanced_accuracy_score(SY, pred_review),
                 "positif_tweet_%": 100 * pred_tweet.mean()})
    del model_tw  # tiap seed dilepas sebelum seed berikutnya
    free_memory()
    print(f"seed {seed} selesai: {memory_report()}")
print("\n" + pd.DataFrame(rows).round(3).to_string(index=False))

seed 0 selesai: VRAM terpakai 0.52 GB (puncak sejak laporan sebelumnya 2.46 GB)
seed 1 selesai: VRAM terpakai 0.52 GB (puncak sejak laporan sebelumnya 2.46 GB)
seed 2 selesai: VRAM terpakai 0.52 GB (puncak sejak laporan sebelumnya 2.47 GB)

 seed  akurasi_tweet  balanced_tweet  balanced_ulasan  positif_tweet_%
    0          0.549           0.644            0.559             80.8
    1          0.482           0.592            0.586             87.5
    2          0.778           0.816            0.691             55.1


Tiga seed, tiga cerita. *Seed* hanya menentukan bobot awal kepala, urutan data, dan dropout. Seed 0 runtuh: 97,0% tweet ditebak positif, dan akurasinya 0,393, di bawah tebakan "selalu negatif". Seed 1 setengah jalan (balanced accuracy 0,613). Seed 2 belajar: balanced accuracy 0,739, di atas model SST-2 (0,655).

Di GPU, bobot awal dan urutan data tetap diambil dari generator acak CPU. Namun, dropout memakai generator acak milik GPU dan pembulatan angkanya sedikit berbeda, jadi hasil tiap seed bisa bergeser jauh: seed yang runtuh bisa lain, atau tidak ada yang runtuh. Tandanya tetap `positif_tweet_%` yang mendekati 0% atau 100%.

**Fine-tuning dengan 1.000 contoh tidak stabil, jadi pembanding pun perlu beberapa seed.** Dengan satu seed yang runtuh saja, kita akan menyimpulkan bahwa melatih di tweet gagal total; dengan satu seed yang berhasil saja, bahwa cara itu jelas unggul, padahal seed terbaik pun baru menutup kurang dari separuh penurunan (0,739, sementara model SST-2 meraih 0,839 di ulasan film).

Tiga baris `seed ... selesai` di atas tabel memastikan setiap model dilepas sebelum seed berikutnya: VRAM terpakai mestinya kembali ke sekitar 0,5 GB, seperti setelah `gpt2-large` dilepas tadi.

### Evaluasi generalisasi tugas

Seberapa baik satu model menangani tugas yang berbeda-beda? Buku memakai empat tugas GLUE: SST-2 (sentimen), QNLI dan MNLI (inferensi), dan QQP (parafrasa: apakah dua pertanyaan bermakna sama).

> **Listing buku -- tujuh error kecil, satu cacat besar**
>
> Error-nya datang dari pustaka yang berubah (`'glue'` kini `nyu-mll/glue`, `evaluation_strategy` kini `eval_strategy`), tugas berpasangan yang diperlakukan seperti SST-2 (kolom, split, dan jumlah label MNLI), serta import dan `compute_metrics` yang tak ditulis. Cacat besarnya muncul setelah semua diperbaiki:
>
> ```python
> eval_results = trainer.evaluate()   # trainer.train() tidak pernah dipanggil
> ```
>
> Yang terukur adalah kepala klasifikasi acak (`epoch` = 0), bukan generalisasi.

Untuk LLM, cara yang lebih wajar: **satu** model tanpa pelatihan per tugas, diminta lewat prompt. Kita uji GPT-2 124M pada 500 contoh validation per tugas.

In [27]:
GLUE_VALIDATION = {"sst2": "validation", "qnli": "validation", "qqp": "validation", "mnli": "validation_matched"}
PROMPTS = {  # (instruksi, pembentuk sisa prompt, kata label menurut urutan id label resmi)
    "sst2": ("Is the sentiment of this movie review positive or negative?\n\nReview:",
             lambda d: f" {d['sentence'].strip()}\nAnswer:", [" negative", " positive"]),
    "qnli": ("Does the sentence contain the answer to the question? Answer yes or no.\n\nQuestion:",
             lambda d: f" {d['question']}\nSentence: {d['sentence']}\nAnswer:", [" yes", " no"]),
    "qqp": ("Are these two questions asking the same thing? Answer yes or no.\n\nQuestion 1:",
            lambda d: f" {d['question1']}\nQuestion 2: {d['question2']}\nAnswer:", [" no", " yes"]),
    "mnli": ("Does the premise imply the hypothesis? Answer yes, maybe, or no.\n\nPremise:",
             lambda d: f" {d['premise']}\nHypothesis: {d['hypothesis']}\nAnswer:", [" yes", " maybe", " no"]),
}
rows = []
for task, (instruction, make_suffix, words) in PROMPTS.items():
    data = load_dataset("nyu-mll/glue", task, split=GLUE_VALIDATION[task]).shuffle(seed=0).select(range(500))  # 500 contoh per tugas
    s = label_scores(instruction, [make_suffix(x) for x in data], [gpt2_tok.encode(w)[0] for w in words])
    y = np.array(data["label"])
    mean, lo, hi = boot_ci(s.argmax(1) == y)
    rows.append({"tugas": task, "akurasi": mean, "CI95": f"{lo:.3f}-{hi:.3f}",
                 "mayoritas": np.bincount(y).max() / len(y),
                 "tebakan_per_label": dict(zip([w.strip() for w in words], np.bincount(s.argmax(1), minlength=len(words)).tolist()))})
print("GPT-2 124M zero-shot, 500 contoh validation per tugas\n")
print(pd.DataFrame(rows).round(3).to_string(index=False))

GPT-2 124M zero-shot, 500 contoh validation per tugas

tugas  akurasi        CI95  mayoritas                   tebakan_per_label
 sst2    0.718 0.678-0.758      0.504  {'negative': 255, 'positive': 245}
 qnli    0.544 0.502-0.588      0.524              {'yes': 86, 'no': 414}
  qqp    0.534 0.492-0.578      0.642             {'no': 270, 'yes': 230}
 mnli    0.412 0.368-0.454      0.356 {'yes': 319, 'maybe': 0, 'no': 181}


**Tanpa pelatihan per tugas, GPT-2 kecil berhasil pada sentimen, paling banter tipis pada inferensi, dan gagal pada parafrasa.** Kuncinya kolom `mayoritas`:

- **SST-2**: 0,718, jauh di atas 0,504.
- **QNLI**: 0,544, tetapi intervalnya memuat baseline 0,524: tak terbedakan dari tebakan mayoritas.
- **QQP**: 0,534, *di bawah* 0,642, lebih buruk daripada selalu menjawab "tidak sama".
- **MNLI**: 0,412, di atas baseline 0,356, tetapi *maybe* tak pernah dipilih: tugas tiga kelas dikerjakan seperti dua kelas.

Ini pun baru satu prompt per tugas, padahal di UTH-2 mengganti contoh saja menggeser akurasi sampai 20 poin.

## Evaluasi continual learning

**Continual learning** berarti model mempelajari tugas baru satu per satu, dan kita ingin ia tetap ingat tugas lama. Buku melatih BERT berurutan pada SST-2, QNLI, lalu QQP, dan mengukur ulang semua tugas setelah tiap tahap. Turunnya akurasi tugas lama disebut **catastrophic forgetting**.

> **Listing buku -- empat error dan satu baris tanpa efek**
>
> Tiga sama dengan listing generalisasi tugas: `'glue'` tanpa namespace (`HfUriError`), `evaluation_strategy` (`TypeError`), dan `compute_metrics` yang tak ditulis (`KeyError: 'eval_accuracy'`). Yang baru: fungsinya tak pernah `return results`, jadi `cl_results.items()` gagal dengan `AttributeError` setelah semua pelatihan selesai. Baris `model.config.num_labels = 3 if task == 'mnli' else 2` pun tak berefek pada kepala klasifikasi.

Versi di bawah memperbaiki semuanya dan menyiapkan dua hal untuk UTH-3: opsi kepala per tugas dan rekaman representasi BERT setelah tiap tugas. Data train penuh ketiga tugas berisi sekitar 536 ribu contoh, jadi tiga epoch berarti lebih dari 1,6 juta contoh dilalui: berhari-hari di CPU dan masih berjam-jam di GPU. Karena itu tiap tugas mendapat 600 contoh train, 300 validation, dan satu epoch.

In [28]:
model_name = "bert-base-uncased"


def accuracy_metrics(p):
    pred = p.predictions.argmax(-1)
    return {"accuracy": float((pred == p.label_ids).mean()),
            "pred1": float((pred == 1).mean())}  # tambahan: proporsi tebakan label 1


def evaluate_continual_learning(model_name, tasks=("sst2", "qnli", "qqp"), num_epochs=1, n_train=600, n_val=300,
                                per_task_heads=False, after_task=None):
    """Fungsi buku setelah diperbaiki: dataset ber-namespace, eval_strategy, compute_metrics, dan return.
    Tambahan: per_task_heads dan after_task, untuk pemeriksaan di bagian berikutnya."""
    transformers.set_seed(0)
    model = AutoModelForSequenceClassification.from_pretrained(model_name)
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    initial_head = copy.deepcopy(model.classifier.state_dict())
    heads, results, share_label1 = {}, {}, {}

    def preprocess_function(examples, task):
        if task == "qqp":
            texts = (examples["question1"], examples["question2"])
        elif task == "qnli":
            texts = (examples["question"], examples["sentence"])
        else:  # sst2
            texts = (examples["sentence"], None)
        tokenized = tokenizer(*texts, padding=True, truncation=True)
        tokenized["labels"] = examples["label"]
        return tokenized

    def tokenized_subset(task, split, n):
        raw = load_dataset("nyu-mll/glue", task, split=split).shuffle(seed=0).select(range(n))
        return raw.map(lambda x: preprocess_function(x, task), batched=True, remove_columns=raw.column_names)

    for task in tasks:
        if per_task_heads:
            model.classifier.load_state_dict(initial_head)  # kontrol: kepala baru untuk setiap tugas
        args = TrainingArguments(output_dir=f"./results_{task}", num_train_epochs=num_epochs, learning_rate=2e-5,
                                 per_device_train_batch_size=16, per_device_eval_batch_size=16, eval_strategy="epoch",
                                 save_strategy="no",  # tanpa ini, transformers 5 menulis checkpoint ~1,2 GB per tugas
                                 disable_tqdm=True, report_to="none", use_cpu=USE_CPU,
                                 dataloader_pin_memory=DEVICE.type == "cuda")
        trainer = Trainer(model=model, args=args, train_dataset=tokenized_subset(task, "train", n_train),
                          eval_dataset=tokenized_subset(task, "validation", n_val), compute_metrics=accuracy_metrics)
        trainer.remove_callback(PrinterCallback)
        trainer.train()
        print(f"  {task}: dilatih di {model_device(model, 'BERT continual')}")
        heads[task] = copy.deepcopy(model.classifier.state_dict())
        if after_task:
            after_task(model, task)

        results[task], share_label1[task] = {}, {}
        for eval_task in tasks[:tasks.index(task) + 1]:
            if per_task_heads:
                model.classifier.load_state_dict(heads[eval_task])
            metrics = trainer.evaluate(eval_dataset=tokenized_subset(eval_task, "validation", n_val))
            results[task][eval_task] = metrics["eval_accuracy"]
            share_label1[task][eval_task] = metrics["eval_pred1"]
        if per_task_heads:
            model.classifier.load_state_dict(heads[task])
        del trainer
        free_memory()
    return results, share_label1


# Untuk pemeriksaan di bagian berikutnya: simpan representasi encoder atas kalimat SST-2 setelah setiap tugas
probe_train = sst_train_full.shuffle(seed=0).select(range(600))  # kalimat train yang sama dengan fine-tuning SST-2
probe_val = sst_val.shuffle(seed=0).select(range(300))            # kalimat evaluasi yang sama
PROBE_FEATURES = {}


@torch.no_grad()
def pooled_features(model, texts, batch_size=64):
    model.eval()
    out = []
    for start in range(0, len(texts), batch_size):
        enc = bert_tok(list(texts[start:start + batch_size]), return_tensors="pt", truncation=True, padding=True,
                       max_length=128).to(model.device)
        out.append(model.bert(**enc).pooler_output.cpu())
    model.train()
    return torch.cat(out).numpy()


def record_probe_features(model, stage):
    PROBE_FEATURES[stage] = (pooled_features(model, list(probe_train["sentence"])),
                             pooled_features(model, list(probe_val["sentence"])))


cl_results, cl_share = evaluate_continual_learning(model_name, after_task=record_probe_features)
free_memory()  # model jalur buku hanya hidup di dalam fungsi; cache GPU-nya dilepas di sini
for task, task_results in cl_results.items():
    print(f"\nAfter training on {task}:")
    for eval_task, accuracy in task_results.items():
        print(f" {eval_task} accuracy: {accuracy:.2f}")

  sst2: dilatih di cuda:0
  qnli: dilatih di cuda:0
  qqp: dilatih di cuda:0

After training on sst2:
 sst2 accuracy: 0.72

After training on qnli:
 sst2 accuracy: 0.62
 qnli accuracy: 0.72

After training on qqp:
 sst2 accuracy: 0.51
 qnli accuracy: 0.49
 qqp accuracy: 0.63


**Sekilas, inilah catastrophic forgetting yang ingin ditunjukkan buku.** Pada run acuan, SST-2 meraih 0,75 tepat setelah dilatih, 0,68 setelah QNLI, lalu **0,51** setelah QQP. QNLI turun dari 0,74 ke **0,49**. Di GPU, angka tiap tahap bisa lain; yang perlu dicocokkan adalah jatuhnya akurasi tugas lama setelah QQP, yang akan kita bedah di UTH-3.

Namun, 0,51 bisa berarti lupa, bisa juga sekadar menebak satu kelas. Ketiga tugas juga berbagi satu kepala klasifikasi, padahal label 1 berarti positif di SST-2, *not_entailment* di QNLI, dan duplikat di QQP. Yang hilang itu pengetahuan di **encoder**, badan BERT di bawah kepala klasifikasi, atau hanya kepalanya yang ditimpa?

### Under the hood (UTH-3): lupa, atau salah membaca?

Setiap akurasi kini dicetak bersama **proporsi tebakan label 1** dan baseline mayoritasnya: model yang menebak satu kelas untuk semua input terlihat dari proporsi 0% atau 100%. Lalu seluruh urutan diulang dengan **kepala per tugas**, disimpan setelah dilatih dan dipasang lagi saat dievaluasi. Bila masalahnya hanya kepala yang ditimpa, SST-2 mestinya selamat.

In [29]:
head_results, head_share = evaluate_continual_learning(model_name, per_task_heads=True)
free_memory()
print(memory_report(), "\n")

TASKS = ["sst2", "qnli", "qqp"]
label1_rate = {t: float(np.mean(load_dataset("nyu-mll/glue", t, split="validation").shuffle(seed=0).select(range(300))["label"]))
               for t in TASKS}


def show_matrix(title, results, share):
    """Akurasi setelah tiap tugas, [proporsi tebakan label 1], plus baseline mayoritas subset evaluasi."""
    table = pd.DataFrame({f"setelah {t}": {e: f"{results[t][e]:.3f} [{share[t][e]:.0%}]" for e in results[t]} for t in TASKS},
                         index=TASKS).fillna("-")
    table["mayoritas"] = [f"{max(label1_rate[e], 1 - label1_rate[e]):.3f}" for e in TASKS]
    table["label 1 asli"] = [f"{label1_rate[e]:.0%}" for e in TASKS]
    bwt = np.mean([results[TASKS[-1]][t] - results[t][t] for t in TASKS[:-1]])
    print(f"{title}   (BWT {bwt:+.3f})\n{table.to_string()}\n")


show_matrix("kepala bersama (buku)", cl_results, cl_share)
show_matrix("kepala per tugas (kontrol)", head_results, head_share)

  sst2: dilatih di cuda:0
  qnli: dilatih di cuda:0
  qqp: dilatih di cuda:0
VRAM terpakai 0.52 GB (puncak sejak laporan sebelumnya 3.38 GB) 

kepala bersama (buku)   (BWT -0.220)
     setelah sst2 setelah qnli setelah qqp mayoritas label 1 asli
sst2  0.723 [49%]  0.620 [85%]  0.510 [0%]     0.510          49%
qnli            -  0.720 [45%]  0.493 [4%]     0.543          54%
qqp             -            -  0.630 [0%]     0.630          37%

kepala per tugas (kontrol)   (BWT -0.232)
     setelah sst2 setelah qnli setelah qqp mayoritas label 1 asli
sst2  0.723 [49%]  0.513 [98%]  0.510 [0%]     0.510          49%
qnli            -  0.730 [48%]  0.480 [2%]     0.543          54%
qqp             -            -  0.630 [0%]     0.630          37%



**Tahap terakhir bukan soal lupa, melainkan runtuh.** Setelah QQP, proporsi tebakan label 1 (angka dalam kurung siku) jatuh ke 0% untuk SST-2 dan QQP, dan 4% untuk QNLI. Karena itu, SST-2 (0,510) dan QQP (0,630) persis sama dengan baseline mayoritasnya, sedangkan QNLI (0,490) di bawahnya, karena kelas yang hampir selalu ditebaknya justru kelas minoritas.

**Kepala per tugas juga tidak menolong**: dengan kepala SST-2 miliknya sendiri pun, SST-2 tetap jatuh ke 0,510. Jadi bukan hanya kepalanya yang ditimpa; encoder di bawahnya ikut bergeser, sampai kepala lama tak lagi cocok dengannya. **BWT** (*backward transfer*) di judul tabel, rata-rata perubahan akurasi tugas lama sampai akhir urutan, sekitar -0,25 untuk kedua versi: tugas lama kehilangan 25 poin.

Namun, encoder yang bergeser belum tentu lupa. **Probe linear**, regresi logistik baru di atas representasi encoder yang disimpan tiap tahap, akan memeriksanya.

In [30]:
pretrained = AutoModelForSequenceClassification.from_pretrained(model_name).to(DEVICE)
model_device(pretrained, "BERT pra-latih")
record_probe_features(pretrained, "pra-latih")  # kontrol: encoder sebelum tugas apa pun
del pretrained
free_memory()

for stage in ["pra-latih", *TASKS]:
    train_x, val_x = PROBE_FEATURES[stage]
    probe = LogisticRegression(max_iter=3000).fit(train_x, probe_train["label"])
    label = "BERT pra-latih" if stage == "pra-latih" else f"versi buku setelah {stage}"
    print(f"probe linear SST-2, {label:<26}: {probe.score(val_x, probe_val['label']):.3f}")

probe linear SST-2, BERT pra-latih            : 0.780
probe linear SST-2, versi buku setelah sst2   : 0.793
probe linear SST-2, versi buku setelah qnli   : 0.837
probe linear SST-2, versi buku setelah qqp    : 0.810


**Sentimen masih terbaca dari encoder; kepalanya yang tak bisa lagi membacanya.** Probe membaca sentimen SST-2 dengan akurasi 0,813 tepat setelah SST-2, dan tetap 0,813 setelah QQP, ketika model yang sama menebak negatif untuk setiap kalimat. Pada BERT pra-latih, probe meraih 0,777: sebagian besar informasi itu sudah ada sebelum fine-tuning.

Jadi matriks akurasi perlu dua pendamping: baseline dan proporsi tebakan untuk mengenali keruntuhan, serta probe untuk membedakan informasi yang hilang dari yang tak terbaca. Kepala yang rusak bisa dilatih ulang, asalkan data tugas lama masih ada.

## Tantangan cross-validation dan praktik terbaik

Buku menutup bab dengan empat tantangan: kontaminasi data, biaya komputasi, pergeseran domain, dan sensitivitas prompt.

### Kontaminasi data

Bila dokumen test punya kembaran di train, skornya mengukur hafalan. Kembaran jarang identik, jadi kita butuh ukuran *seberapa mirip*. Potong *"the cat sat on the mat"* dan *"the cat sat on a mat"* menjadi **shingle** tiga kata (`the cat sat`, `cat sat on`, ...). Dari enam shingle berbeda, dua dimiliki keduanya, jadi **kemiripan Jaccard**-nya 2/6: irisan dibagi gabungan.

Membandingkan setiap pasangan dari 95.932 dokumen terlalu mahal. **MinHash** meringkas shingle tiap dokumen menjadi 128 angka untuk menaksir Jaccard, dan **LSH** (*locality-sensitive hashing*) hanya mempertemukan dokumen yang ringkasannya mirip. Fungsi buku menganggap setiap kandidat LSH, yang kemungkinan besar berkemiripan di atas 0,8, sebagai duplikat.

> **Listing buku -- `get_ngrams` tidak pernah ditulis**
>
> Listing berhenti dengan `NameError`, padahal fungsi inilah yang menentukan arti "mirip". Kita memakai shingle lima kata berhuruf kecil; sisa fungsi buku dipakai apa adanya.

In [31]:
# CPU: datasketch (MinHash dan LSH) tidak memakai GPU.
from datasketch import MinHash, MinHashLSH


def get_ngrams(text, n=5):
    """Pelengkap yang tidak ditulis buku: shingle 5 kata huruf kecil; teks di bawah 5 kata menjadi satu shingle."""
    words = text.lower().split()
    return [" ".join(words[i:i + n]) for i in range(max(1, len(words) - n + 1))]


def minhash_of(text):
    """MinHash 128 permutasi atas shingle yang sama; dipakai sel-sel sesudah ini."""
    m = MinHash(num_perm=128)
    m.update_batch([g.encode("utf8") for g in get_ngrams(text)])
    return m


def deduplicate_data(texts, threshold=0.8):
    # Initialize LSH index for fast similarity search
    lsh = MinHashLSH(threshold=threshold, num_perm=128)
    unique_texts = []
    for idx, text in enumerate(texts):
        minhash = MinHash(num_perm=128)
        for ngram in get_ngrams(text):
            minhash.update(ngram.encode('utf8'))
        if not lsh.query(minhash):  # Check if similar text exists
            lsh.insert(str(idx), minhash)
            unique_texts.append(text)
    return unique_texts


unique_texts = deduplicate_data(list(CORPUS["text"]))
print(f"{len(CORPUS)} -> {len(unique_texts)} teks ({len(CORPUS) - len(unique_texts)} dibuang)")
print("tipe nilai kembali:", type(unique_texts).__name__, "berisi", type(unique_texts[0]).__name__)

95932 -> 95379 teks (553 dibuang)
tipe nilai kembali: list berisi str


MinHash membuang **553** dari 95.932 dokumen. Baris kedua memperlihatkan keterbatasan fungsi buku: fungsi itu hanya mengembalikan daftar teks, tanpa indeks atau domain. Untuk tahu dokumen *mana* yang dibuang, sel berikut merekonstruksi indeksnya dari urutan teks, lalu memecah hasilnya per domain.

In [32]:
def kept_mask(texts, unique):
    """Dokumen mana yang dipertahankan? Fungsi buku hanya mengembalikan teks (urutannya terjaga), jadi indeksnya direkonstruksi."""
    keep, j = np.zeros(len(texts), bool), 0
    for i, text in enumerate(texts):
        if j < len(unique) and text == unique[j]:
            keep[i] = True
            j += 1
    assert j == len(unique)
    return keep


keep = kept_mask(list(CORPUS["text"]), unique_texts)
normalized = CORPUS["text"].str.lower().str.split().str.join(" ")
exact_dup = normalized.duplicated()  # kembar persis setelah huruf dikecilkan dan spasi dirapikan
table = pd.DataFrame({"dokumen": counts,
                      "dibuang": CORPUS[~keep]["domain"].value_counts(),
                      "di_antaranya_kembar_persis": CORPUS[~keep & exact_dup]["domain"].value_counts()})
table = table.fillna(0).astype(int).loc[counts.index]
table["persen_dibuang"] = (100 * table["dibuang"] / table["dokumen"]).round(2)
print(table.to_string())
# Satu near-duplicate MMLU yang dibuang, beserta dokumen sebelumnya yang paling mirip (Jaccard atas shingle 5 kata)
example = np.where(~keep & ~exact_dup & (CORPUS["domain"] == "ujian (MMLU)"))[0][0]
example_shingles = set(get_ngrams(CORPUS["text"][example]))
earlier = np.where((CORPUS["domain"] == "ujian (MMLU)") & (CORPUS.index < example))[0]
jaccard, partner = max((len(example_shingles & set(get_ngrams(CORPUS["text"][j]))) /
                        len(example_shingles | set(get_ngrams(CORPUS["text"][j]))), j) for j in earlier)
print("\nnear-duplicate MMLU yang dibuang:", repr(CORPUS["text"][example][:105]))
print(f"dokumen yang dianggap kembarannya: {CORPUS['text'][partner][:105]!r}  (Jaccard {jaccard:.3f})")

                         dokumen  dibuang  di_antaranya_kembar_persis  persen_dibuang
domain                                                                               
tweet (TweetEval)          45615      196                          30            0.43
ensiklopedia (wikitext)    17556       89                          86            0.51
ujian (MMLU)               15858      255                         142            1.61
matematika (GSM8K)          8792        1                           0            0.01
sains sekolah (ARC)         7787       12                           3            0.15
kode (HumanEval)             164        0                           0            0.00
chat (MT-Bench)              160        0                           0            0.00

near-duplicate MMLU yang dibuang: 'Find the sum of the given polynomials in the given polynomial ring. f(x) = 4x - 5, g(x) = 2x^2 - 4x + 2 i'
dokumen yang dianggap kembarannya: 'Find the product of the given polynomials in th

**"Duplikat" ternyata bukan satu hal.** Di ensiklopedia, 86 dari 89 dokumen yang dibuang adalah **kembar persis**: identik setelah huruf dikecilkan dan spasi dirapikan. Di tweet hanya 30 dari 196; sisanya **near-duplicate**, mirip tetapi tidak identik.

Deteksi near-duplicate bisa salah tangkap. Lihat dua baris terakhir output: soal MMLU yang dibuang meminta *jumlah* dua polinom, kembarannya meminta *hasil kali* polinom yang sama. Hanya satu kata berbeda, jadi Jaccard-nya 0,838, padahal jawabannya berbeda. **MinHash mengukur tumpang tindih teks, bukan kesamaan makna.**

### Under the hood (UTH-4): kebocoran yang tak terlihat oleh split

`deduplicate_data` bekerja di dalam satu daftar, padahal kebocoran yang merusak evaluasi justru melintasi split: dokumen test yang kembarannya ada di train. Kita periksa split bertingkat dari awal bab, lalu split resmi QQP dari GLUE.

In [33]:
# CPU: datasketch (MinHash dan LSH) tidak memakai GPU.
lsh = MinHashLSH(threshold=0.8, num_perm=128)
for i in corpus_train.index:
    lsh.insert(str(i), minhash_of(CORPUS["text"][i]))
has_near_twin = np.array([bool(lsh.query(minhash_of(CORPUS["text"][i]))) for i in corpus_test.index])
train_texts = set(normalized[corpus_train.index])
has_exact_twin = np.array([normalized[i] in train_texts for i in corpus_test.index])
del lsh

leak = pd.DataFrame({"dokumen_test": corpus_test["domain"].value_counts(),
                     "kembar_persis_di_train": corpus_test[has_exact_twin]["domain"].value_counts(),
                     "near_duplicate_di_train": corpus_test[has_near_twin]["domain"].value_counts()})
leak = leak.fillna(0).astype(int).loc[counts.index]
print(leak.to_string())
print(f"\ntotal: {has_exact_twin.sum()} kembar persis dan {has_near_twin.sum()} near-duplicate"
      f" dari {len(corpus_test)} dokumen test ({has_near_twin.mean():.2%})")

                         dokumen_test  kembar_persis_di_train  near_duplicate_di_train
domain                                                                                
tweet (TweetEval)                4562                       3                       31
ensiklopedia (wikitext)          1756                      11                       12
ujian (MMLU)                     1586                      18                       34
matematika (GSM8K)                879                       0                        0
sains sekolah (ARC)               779                       0                        2
kode (HumanEval)                   16                       0                        0
chat (MT-Bench)                    16                       0                        0

total: 32 kembar persis dan 79 near-duplicate dari 9594 dokumen test (0.82%)


Setelah split bertingkat, **79** dokumen test (0,82%) punya near-duplicate di train. Kecil, tetapi tidak merata: MMLU paling parah, 34 dari 1.586 dokumen test, sedangkan GSM8K nol karena soalnya memang nyaris tak punya kembaran.

Pelajarannya: deduplikasi seluruh data **sebelum** membagi, atau periksa kembaran test di train **sesudah** membagi.

Split resmi sebuah benchmark pun bisa bocor. Di QQP, satu pertanyaan Quora sering muncul di banyak pasangan. Seberapa jauh label validation bisa ditebak hanya dengan mengingat train?

In [34]:
# CPU: hanya pandas dan himpunan Python.
qqp = load_dataset("nyu-mll/glue", "qqp")
qqp_train, qqp_val = qqp["train"].to_pandas(), qqp["validation"].to_pandas()
for frame in (qqp_train, qqp_val):
    frame["q1"] = frame["question1"].map(lambda s: " ".join(str(s).lower().split()))
    frame["q2"] = frame["question2"].map(lambda s: " ".join(str(s).lower().split()))
train_questions = set(qqp_train["q1"]) | set(qqp_train["q2"])
val_questions = pd.concat([qqp_val["q1"], qqp_val["q2"]]).drop_duplicates()
print(f"QQP: {len(qqp_train)} pasangan train, {len(qqp_val)} pasangan validation")
print(f"pertanyaan validation (unik) yang muncul persis di train : {val_questions.isin(train_questions).mean():.1%}")
both_in_train = qqp_val["q1"].isin(train_questions) & qqp_val["q2"].isin(train_questions)
print(f"pasangan validation yang kedua pertanyaannya ada di train: {both_in_train.mean():.1%}")

# "Model" penghafal: gabungkan pasangan duplikat train menjadi klaster (union-find), lalu ikuti transitivitasnya
parent = {}


def root(x):
    while parent.get(x, x) != x:
        parent[x] = parent.get(parent[x], parent[x])
        x = parent[x]
    return x


for a, b, label in zip(qqp_train["q1"], qqp_train["q2"], qqp_train["label"]):
    if label == 1 and root(a) != root(b):
        parent[root(a)] = root(b)

same_cluster = np.array([a in train_questions and b in train_questions and root(a) == root(b)
                         for a, b in zip(qqp_val["q1"], qqp_val["q2"])])
majority = int(qqp_train["label"].mode()[0])
y_val = qqp_val["label"].to_numpy()
memorizer = np.where(same_cluster, 1, majority)
print(f"\npenghafal tanpa parameter: akurasi {(memorizer == y_val).mean():.3f} | baseline mayoritas {(y_val == majority).mean():.3f}")
print(f"pasangan yang ditandai duplikat: {same_cluster.mean():.1%}, presisinya {y_val[same_cluster].mean():.3f}")
del qqp, qqp_train, qqp_val, train_questions, val_questions, parent
free_memory()

QQP: 363846 pasangan train, 40430 pasangan validation
pertanyaan validation (unik) yang muncul persis di train : 40.7%
pasangan validation yang kedua pertanyaannya ada di train: 31.5%

penghafal tanpa parameter: akurasi 0.850 | baseline mayoritas 0.632
pasangan yang ditandai duplikat: 21.9%, presisinya 0.998


Kebocorannya berupa **pertanyaan yang dipakai ulang**: 40,7% pertanyaan unik di validation muncul persis di train.

Kebocoran itu cukup untuk membangun "model" tanpa parameter, sebuah **penghafal**. Bila train menyatakan A dan B duplikat, lalu B dan C duplikat, penghafal menyimpulkan A dan C pun duplikat; pasangan lain ditebak kelas mayoritas. Akurasinya **0,850**, jauh di atas baseline 0,632, dan tebakan *duplikat*-nya benar 998 dari 1.000 kali.

**Penghafal ini tidak membaca satu kata pun.** Model yang di-fine-tune pada QQP bisa memakai jalan pintas yang sama tanpa ketahuan, jadi sebagian akurasinya mungkin mengukur ingatan. Split resmi dibuat per pasangan; split per pertanyaan akan memperlihatkannya.

### Biaya komputasi

Untuk menekan biaya, buku menyarankan stratified sampling atau validasi satu split, lalu memberi listing `create_efficient_splits` sebagai contoh. Fungsinya berjalan tanpa error, jadi kita periksa dua hal: apakah hasilnya bisa diulang, dan apakah fungsi itu memang hemat.

In [35]:
# CPU: pembagian fold dengan scikit-learn, tanpa model.
from collections import defaultdict


def create_efficient_splits(data, labels, n_splits=5):
    # Group data by domain
    domain_data = defaultdict(list)
    for text, domain in zip(data, labels):
        domain_data[domain].append(text)
    # Create stratified splits
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True)
    splits = []
    for train_idx, val_idx in skf.split(data, labels):
        splits.append((train_idx, val_idx))
    return splits


def fold_of(splits, n):
    """Untuk setiap dokumen: di fold validasi mana ia ditempatkan."""
    fold = np.zeros(n, int)
    for k, (_, val_idx) in enumerate(splits):
        fold[val_idx] = k
    return fold


texts, domains = list(CORPUS["text"]), list(CORPUS["domain"])
np.random.seed(0)  # fungsi buku memakai RNG global numpy; seed ini hanya membuat output sel ini sama setiap kali dijalankan
book_a = fold_of(create_efficient_splits(texts, domains), len(texts))
book_b = fold_of(create_efficient_splits(texts, domains), len(texts))
seeded = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
fixed_a = fold_of(list(seeded.split(texts, domains)), len(texts))
fixed_b = fold_of(list(seeded.split(texts, domains)), len(texts))
print(f"fungsi buku, dua panggilan identik  : {np.mean(book_a == book_b):.1%} dokumen mendarat di fold yang sama")
print(f"dengan random_state=0, dua panggilan: {np.mean(fixed_a == fixed_b):.1%}")

fungsi buku, dua panggilan identik  : 19.8% dokumen mendarat di fold yang sama
dengan random_state=0, dua panggilan: 100.0%


**Fungsi buku memberi split berbeda setiap kali dipanggil.** Dua panggilan identik hanya menaruh 19,8% dokumen di fold yang sama, nyaris sama dengan peluang acak (20%), karena `shuffle=True` dipakai tanpa `random_state`. Dengan `random_state=0`, keduanya sepakat 100%.

Klaim hematnya juga terbalik: 5-fold berarti lima kali pelatihan. Di UTH-1, biaya itu membeli varians 11 kali lebih kecil; sering layak, tetapi tetap pengeluaran.

### Pergeseran domain

Buku menyarankan evaluasi tiap domain secara terpisah, agar kelemahan satu domain tidak tertutup rata-rata.

> **Listing buku -- skor yang tidak mengukur kinerja**
>
> ```python
> outputs = model(inputs)
> score = outputs.logits.mean().item()
> ```
>
> `model(inputs)` gagal dengan `AttributeError`, sama seperti di bagian adaptasi domain. Setelah diperbaiki, skornya adalah rata-rata *semua* logit atas 50.257 token kosakata GPT-2. Softmax tidak berubah bila semua logit digeser bersama, jadi rata-rata itu justru bagian yang tidak memengaruhi tebakan model.

Kita catat juga **loss per token** (dalam *nats*): makin rendah, makin baik model menebak teks domain itu. Kita pakai hingga 60 dokumen test per domain; kode dan chat hanya punya 16.

In [36]:
def evaluate_domain_performance(model, tokenizer, eval_data):
    """Listing buku dengan model(**inputs); di forward yang sama dicatat juga loss per token."""
    domain_scores, domain_loss = defaultdict(list), defaultdict(list)
    for text, domain in eval_data:
        inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=256).to(model.device)  # maksimal 256 token
        if inputs["input_ids"].shape[1] < 2:
            continue
        with torch.no_grad():
            outputs = model(**inputs, labels=inputs["input_ids"])
        domain_scores[domain].append(outputs.logits.mean().item())  # skor versi buku
        domain_loss[domain].append(outputs.loss.item())
    return ({d: np.mean(v) for d, v in domain_scores.items()},
            {d: np.mean(v) for d, v in domain_loss.items()})


eval_data = [(text, domain) for domain, group in CORPUS.loc[corpus_test.index].groupby("domain")
             for text in group["text"].sample(min(60, len(group)), random_state=0)]  # hingga 60 per domain (kode dan chat hanya 16)
score, loss = evaluate_domain_performance(gpt2, gpt2_tok, eval_data)
table = pd.DataFrame({"skor_buku_logits_mean": score, "loss_nats_per_token": loss}).sort_values("loss_nats_per_token")
table["peringkat_skor"] = table["skor_buku_logits_mean"].rank(ascending=False).astype(int)
table["peringkat_loss"] = table["loss_nats_per_token"].rank().astype(int)
print(f"{len(eval_data)} dokumen test dari {table.shape[0]} domain\n")
print(table.round(3).to_string())
rho = table["skor_buku_logits_mean"].rank().corr(table["loss_nats_per_token"].rank())
print(f"\nkorelasi peringkat (Spearman) skor buku vs loss: {rho:+.2f}")

332 dokumen test dari 7 domain

                         skor_buku_logits_mean  loss_nats_per_token  peringkat_skor  peringkat_loss
kode (HumanEval)                       -96.258                2.809               1               1
matematika (GSM8K)                    -110.684                3.423               7               2
ujian (MMLU)                          -103.298                3.642               5               3
chat (MT-Bench)                        -96.591                3.797               2               4
sains sekolah (ARC)                   -102.851                3.810               3               5
ensiklopedia (wikitext)               -103.803                4.185               6               6
tweet (TweetEval)                     -103.286                5.256               4               7

korelasi peringkat (Spearman) skor buku vs loss: -0.18


Bila skor buku mengukur kinerja, domain dengan loss terendah mestinya mendapat skor tertinggi, dan korelasi peringkatnya mendekati -1. Kenyataannya **-0,18**. Matematika (GSM8K) punya loss terendah kedua (3,423), tetapi skor buku menaruhnya paling buruk.

Loss juga punya arti langsung: untuk kode (2,809), GPT-2 seolah memilih di antara sekitar 17 token yang sama mungkinnya ($e^{2{,}809} \approx 16{,}6$); untuk tweet (5,256), sekitar 192. **Gagasan evaluasi per domain tetap benar, tetapi angkanya harus loss atau metrik tugas yang bermakna, bukan rata-rata logit.**

### Sensitivitas prompt

Satu prompt hanyalah satu cara bertanya. Buku menyarankan menggabungkan jawaban beberapa variasi prompt: **ensemble**.

> **Listing buku -- ensemble tanpa cara menggabungkan**
>
> ```python
> output = model.generate(inputs, max_length=100)
> return aggregate_responses(responses)
> ```
>
> `generate(inputs)` gagal dengan `AttributeError`. Setelah diperbaiki, fungsi berhenti di `aggregate_responses` (`NameError`), padahal penggabungan itulah inti ensemble.

Mari kita lihat dulu apa yang akan digabungkan: lanjutan ketiga variasi untuk satu ulasan SST-2.

In [37]:
BASE = "Classify the sentiment of this movie review as positive or negative"
review = SX[0]
variants = [f"{BASE}: {review}", f"Please {BASE.lower()}: {review}", f"I want you to {BASE.lower()}: {review}"]
print("ulasan:", repr(review), "| label:", NAME[SY[0]], "\n")
for prompt in variants:
    inputs = gpt2_tok(prompt, return_tensors="pt").to(gpt2.device)
    with torch.no_grad():
        output = gpt2.generate(**inputs, max_length=100, pad_token_id=gpt2_tok.eos_token_id)  # perbaikan: **inputs
    print(repr(gpt2_tok.decode(output[0][inputs["input_ids"].shape[1]:])[:110]))

ulasan: "it 's a charming and often affecting journey ." | label: Positive 

" . . and it's a great movie. It's a great movie. It's a great movie. It's a great movie. It's a great movie. I"
" It's a very well written and well acted film. It's a very well written and well acted film. It's a very well "
" . . but it's also a very sad and depressing movie. It's a sad movie, but it's also a sad movie. It's a sad mo"


Tiga variasi, tiga lanjutan bebas: dua memuji, yang ketiga berbelok ke *"very sad and depressing"*. Tak satu pun berupa label yang bisa diambil suara terbanyaknya.

**Ensemble butuh keluaran yang bisa digabungkan secara numerik.** Sel berikut menilai ketiganya dengan `label_scores` pada 872 kalimat SST-2, lalu merata-ratakan selisih log-probabilitas *positive* dan *negative*.

In [38]:
VARIANT_PREFIXES = [f"{BASE}:", f"Please {BASE.lower()}:", f"I want you to {BASE.lower()}:"]
LOWER_LABEL_IDS = [gpt2_tok.encode(" negative")[0], gpt2_tok.encode(" positive")[0]]
variant_scores = [label_scores(prefix, [f" {x}\nSentiment:" for x in SX], LOWER_LABEL_IDS) for prefix in VARIANT_PREFIXES]

for prefix, s in zip(VARIANT_PREFIXES, variant_scores):
    pred = (s[:, 1] > s[:, 0]).astype(int)
    mean, lo, hi = boot_ci(pred == SY)
    print(f"{prefix[:-1][:40]:<42} akurasi {mean:.3f} [CI95 {lo:.3f}-{hi:.3f}] | ditebak positif {pred.mean():.1%}")
margin = np.mean([s[:, 1] - s[:, 0] for s in variant_scores], axis=0)  # ensemble: rata-rata selisih log-prob
mean, lo, hi = boot_ci((margin > 0).astype(int) == SY)
ensemble_positive = (margin > 0).mean()
print(f"{'ensemble (rata-rata ketiganya)':<42} akurasi {mean:.3f} [CI95 {lo:.3f}-{hi:.3f}] | ditebak positif {ensemble_positive:.1%}")
decisions = [(s[:, 1] > s[:, 0]) for s in variant_scores]
print(f"\nkeputusan varian 2 dan 3 sama dengan varian 1 pada {np.mean(decisions[1] == decisions[0]):.1%}"
      f" dan {np.mean(decisions[2] == decisions[0]):.1%} kalimat")

Classify the sentiment of this movie rev   akurasi 0.585 [CI95 0.554-0.617] | ditebak positif 87.8%
Please classify the sentiment of this mo   akurasi 0.569 [CI95 0.538-0.601] | ditebak positif 91.5%
I want you to classify the sentiment of    akurasi 0.600 [CI95 0.568-0.631] | ditebak positif 80.2%
ensemble (rata-rata ketiganya)             akurasi 0.585 [CI95 0.553-0.616] | ditebak positif 86.9%

keputusan varian 2 dan 3 sama dengan varian 1 pada 96.3% dan 92.3% kalimat


Ketiga variasi meraih 0,569-0,600 dan semuanya condong ke *positive* (80,2%-91,5% tebakan positif, padahal yang positif 50,9%). Ensemble-nya 0,585, tidak lebih baik daripada variasi terbaik.

Sebabnya di baris terakhir: variasi 2 dan 3 memutuskan sama dengan variasi 1 untuk 96,3% dan 92,3% kalimat. **Merata-ratakan prompt yang berkorelasi tidak menghapus bias bersamanya.** Format `Input:`/`Output:` dari listing zero-shot, yang benar-benar berbeda, meraih 0,694. Variasi yang layak di-ensemble adalah yang berbeda format, kata label, dan contohnya.

### Pipeline lengkap

Terakhir, buku merangkai deduplikasi, split, dan evaluasi per domain menjadi satu pipeline.

> **Listing buku -- label yang tergelincir setelah deduplikasi**
>
> ```python
> clean_data = deduplicate_data(data)
> splits = create_efficient_splits(clean_data, domains)
> ```
>
> `clean_data` menyusut sementara `domains` tidak, sehingga `StratifiedKFold` gagal dengan `ValueError`. Lebih mendasar lagi, `splits` tak pernah dipakai: pipeline ini tidak pernah melakukan cross-validation.

**Error itu sebenarnya keberuntungan.** Tanpanya, `zip(clean_data, domains)` akan memasangkan teks dan domain menurut posisi tanpa protes. Sel berikut mengukur akibatnya, lalu memperbaikinya.

In [39]:
# CPU: deduplikasi datasketch dan split scikit-learn, tanpa model.
# Sub-korpus kecil: semua dokumen yang punya kembaran persis + 1.000 dokumen lain
has_twin = normalized.duplicated(keep=False)
sub = pd.concat([CORPUS[has_twin], CORPUS[~has_twin].sample(1000, random_state=0)])
clean_texts = deduplicate_data(list(sub["text"]))
paired = list(zip(clean_texts, sub["domain"]))  # yang dilakukan zip(clean_data, domains) di pipeline buku
true_domains = list(sub["domain"][kept_mask(list(sub["text"]), clean_texts)])
wrong = sum(pair_domain != true for (_, pair_domain), true in zip(paired, true_domains))
print(f"sub-korpus {len(sub)} dokumen -> {len(clean_texts)} setelah deduplikasi, tetapi daftar domain tetap {len(sub)}")
print(f"zip memasangkan {wrong} dari {len(clean_texts)} teks dengan domain yang salah")


def deduplicate_indices(texts, threshold=0.8):
    """Perbaikan: kembalikan posisi dokumen yang dipertahankan, sehingga label apa pun bisa ikut dipilih."""
    lsh = MinHashLSH(threshold=threshold, num_perm=128)
    keep_positions = []
    for idx, text in enumerate(texts):
        m = minhash_of(text)
        if not lsh.query(m):
            lsh.insert(str(idx), m)
            keep_positions.append(idx)
    return keep_positions


clean_sub = sub.iloc[deduplicate_indices(list(sub["text"]))]
print(f"\nversi perbaikan: {len(clean_sub)} dokumen, teks sama dengan versi buku: {clean_sub['text'].tolist() == clean_texts}")
import warnings
with warnings.catch_warnings(record=True) as caught:  # tangkap peringatan sklearn agar yang tercetak hanya pesannya
    warnings.simplefilter("always")
    folds = list(StratifiedKFold(5, shuffle=True, random_state=0).split(clean_sub["text"], clean_sub["domain"]))
print(f"StratifiedKFold kini berjalan: {len(folds)} fold, fold validasi pertama {len(folds[0][1])} dokumen")
for w in caught:
    print("peringatan sklearn:", w.message)

sub-korpus 1484 dokumen -> 1223 setelah deduplikasi, tetapi daftar domain tetap 1484
zip memasangkan 886 dari 1223 teks dengan domain yang salah

versi perbaikan: 1223 dokumen, teks sama dengan versi buku: True
StratifiedKFold kini berjalan: 5 fold, fold validasi pertama 245 dokumen
peringatan sklearn: The least populated class in y has only 1 members, which is less than n_splits=5.


Deduplikasi menyusutkan 1.484 dokumen menjadi 1.223, tetapi daftar domain tetap 1.484. Akibatnya, `zip` memasangkan **886 dari 1.223** teks dengan domain keliru: setiap dokumen yang dibuang menggeser semua label sesudahnya.

Perbaikannya, deduplikasi mengembalikan **posisi** dokumen agar teks dan domain dipilih bersama. Teks yang dipertahankan sama persis dengan versi buku (`True` di output), domainnya kini benar, dan split bertingkat berjalan. Peringatan sklearn di baris terakhir menggemakan awal bab: ada domain berisi satu dokumen, yang sebaiknya dimasukkan seluruhnya ke train.

Kelima fold ini baru bahan: cross-validation sesungguhnya masih harus melatih dan mengevaluasi per domain di setiap fold, seperti yang kita lakukan pada CB.

## Summary

Seluruh bab menguji satu pertanyaan: seberapa yakin kita pada angka evaluasi?

- **Cara membagi data menentukan apa yang diukur.** Stratifikasi menjaga domain langka; split waktu memperlihatkan penurunan yang disembunyikan split acak.
- **Satu angka hanyalah satu sampel.** Split tunggal, contoh few-shot, dan seed fine-tuning sama-sama berayun lebar; laporkan sebarannya.
- **Bandingkan dengan baseline yang tepat.** Selisih akurasi bisa menutupi penurunan lintas domain yang baru terlihat pada balanced accuracy, dan "lupa" pada continual learning bisa jadi hanyalah model yang runtuh ke satu kelas, sementara encoder-nya masih menyimpan sentimen.
- **Kebocoran tidak terlihat oleh split.** Penghafal tanpa parameter meraih 0,850 di QQP.

### What comes next

Probe linear tadi membaca isi encoder terlepas dari jawaban kepalanya; itulah contoh kecil alat **interpretability** (interpretabilitas). Chapter 16 membahasnya secara utuh: memahami dan menjelaskan perilaku LLM.